Goal: Analyse the OSDAR23 dataset and create the new datasets

# All the required libraries

In [ ]:
#Environment
%env OMP_NUM_THREADS=6
import os
os.environ["OMP_NUM_THREADS"] = "6"

# Standards
import ast
import json
import math
import pathlib
import random
import re
import shutil
import warnings
from collections import Counter, defaultdict
from datetime import datetime
from decimal import Decimal
import numpy as np
import pandas as pd
from glob import glob
from tqdm import tqdm
import folium
from IPython.display import display

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from matplotlib.transforms import Affine2D
from matplotlib.widgets import LassoSelector
import plotly.express as px

# Image
import cv2
from PIL import Image, ImageStat
from easyimages import EasyImage, EasyImageList, bbox
from easyimages.easyimages import CTX


# SciPy 
from scipy.interpolate import interp1d
from scipy.optimize import brentq
from scipy.stats import entropy
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from pathlib import Path
from itertools import product



warnings.simplefilter(action="ignore", category=FutureWarning)
%matplotlib inline

# Data path
rail_data_path = "./rail_data/DB/"


# Create dataframe with all data and labels from OSDAR23 dataset (https://data.fid-move.de/dataset/osdar23)

Some checks: 

In [ ]:
glob(f"{rail_data_path}{os.listdir(rail_data_path)[0]}/*.json")[0] #glob() finds all .json files inside the subdirectory os.listdir(rail_data_path)[0] inside rail_data_path-->only one json file in each folder exist

Total number of label IDs in the dataset: 250597:

In [ ]:
num_labels_uids = 0
for d in os.listdir(rail_data_path):
    file_name = glob(f"{rail_data_path}{d}/*.json")[0]
    with open(file_name) as f:
        raw = f.read() #reads content of json file
        num_labels_uids += len(re.findall("uid", raw))
num_labels_uids

In the 3 novatel folders(per station) the txt files with sensor information are included

novatel_oem7_corrimu:  frame_idx,timestamp, gps_week_number,  gps_week_milliseconds,  pitch_rate,roll_rate,yaw_rate,lateral_acc,longitudinal_acc,vertical_acc
novatel_oem7_inspva:   frame_idx,timestamp,latitude,longitude,height,north_velocity,east_velocity,up_velocity,roll,pitch,azimuth
novatel_oem7_insstdev: frame_idx,timestamp,latitude_stdev,longitude_stdev,height_stdev,north_velocity_stdev,east_velocity_stdev,up_velocity_stdev,roll_stdev,pitch_stdev,azimuth_stdev

Function to parse the data in the dataframe

Key Columns

- **path**: Full file path to the image.  
- **timestamp**: Capture time of the image.  
- **sensor**: Type of sensor that produced the data.  
- **dataset**: Same as the `tag` column, groups the data into subsets.  
- **label_type**: Annotation style (e.g., polyline, bounding box, polygon).  
- **val**: Encoded polyline values, convertible into (x, y) coordinates.  
- **closed**: Indicates whether a polyline forms a closed shape.  
- **occlusion**: Visibility level of the annotated object in the image.  
- **railSide**: Specifies whether the annotation belongs to the left or right rail.  
- **object_uid**: Unique object identifier; the same object can appear in multiple images.  
- **longitude**: GPS longitude coordinate.  
- **latitude**: GPS latitude coordinate.  


In [ ]:
def data_to_dataframe():
    """
    Reads JSON files for each station of the dataset and extracts information for:
    - GPS 
    - IMU data (from novatel_oem7_inspva and novatel_oem7_corrimu)
    - Sensors
    - Images
    - Labels
    
    """

    data = []

    # Iterate over all directories in the dataset and open json files
    for d in os.listdir(rail_data_path):
        file_name = glob(f"{rail_data_path}{d}/*.json")[0]  # Get first JSON file
        print(f"Processing JSON: {file_name}")

        with open(file_name) as json_file:
            raw = json.load(json_file)
        raw = raw["openlabel"]

        # Extract data from sensors
        sensor_to_wh = {}
        for k, v in raw["streams"].items():
            if "stream_properties" in v.keys() and "intrinsics_pinhole" in v["stream_properties"]:
                sensor_to_wh[k] = {
                    "width": v["stream_properties"]["intrinsics_pinhole"]["width_px"], 
                    "height": v["stream_properties"]["intrinsics_pinhole"]["height_px"],
                }
            else:
                sensor_to_wh[k] = {"width": np.nan, "height": np.nan}

        # Extract labels
        for k, v in raw["frames"].items():
            frame = k.rjust(3, "0")  # Ensure frame index has 3 digits

            # Load GPS Data (novatel_oem7_inspva)
            csv_inspva = glob(f"{rail_data_path}{d}/novatel_oem7_inspva/{frame}_*.csv")
            if csv_inspva:
                df_inspva = pd.read_csv(csv_inspva[0])
                lati = df_inspva["latitude"].values[0]
                long = df_inspva["longitude"].values[0]
                
            else:
                print(f"Missing novatel_oem7_inspva CSV for frame {frame} in dataset {d}")
                long, lati = np.nan, np.nan
                df_inspva = None

            # Load IMU Data (novatel_oem7_corrimu)
            csv_corrimu = glob(f"{rail_data_path}{d}/novatel_oem7_corrimu/{frame}_*.csv")
            if csv_corrimu:
                df_corrimu = pd.read_csv(csv_corrimu[0])
                imu_data = df_corrimu.iloc[0].to_dict()  # Get first row as dictionary
            else:
                print(f"Missing novatel_oem7_corrimu CSV for frame {frame} in dataset {d}")
                imu_data = {
                    "pitch_rate": np.nan,
                    "roll_rate": np.nan,
                    "yaw_rate": np.nan,
                    "lateral_acc": np.nan,
                    "longitudinal_acc": np.nan,
                    "vertical_acc": np.nan
                }

            for k2, v2 in v["objects"].items():
                for k3, v3 in v2["object_data"].items():
                    for b in v3:
                        holder = {}

                        # Add general metadata
                        holder["longitude"] = long
                        holder["latitude"] = lati
                        holder["tag"] = raw["metadata"]["tagged_file"]
                        holder["type"] = raw["objects"][k2]["type"]
                        holder["name"] = b["name"]
                        holder["label_uid"] = b["uid"]
                        holder["object_uid"] = k2
                        holder["sensor"] = b["coordinate_system"]

                        # Add sensor width and height
                        holder["height"] = sensor_to_wh.get(b["coordinate_system"], {}).get("height", np.nan)
                        holder["width"] = sensor_to_wh.get(b["coordinate_system"], {}).get("width", np.nan)

                        # Add file path
                        holder["path"] = f"{d}{v['frame_properties']['streams'][b['coordinate_system']]['uri']}"
                        holder["dataset"] = d
                        holder["timestamp"] = v["frame_properties"]["timestamp"]
                        holder["label_type"] = k3
                        holder["tag"] = raw["metadata"]["tagged_file"]

                        # Add IMU data
                        if df_corrimu is not None:
                            holder["pitch_rate"] = float(df_corrimu["pitch_rate"].values[0])
                            holder["roll_rate"] = float(df_corrimu["roll_rate"].values[0])
                            holder["yaw_rate"] = float(df_corrimu["yaw_rate"].values[0])
                            holder["lateral_acc"] = float(df_corrimu["lateral_acc"].values[0])
                            holder["longitudinal_acc"] = float(df_corrimu["longitudinal_acc"].values[0])
                            holder["vertical_acc"] = float(df_corrimu["vertical_acc"].values[0])#holder.update(imu_data)

                        # Special case for "track" objects with poly2d labels
                        if holder["type"] == "track" and k3 == "poly2d":
                            holder["closed"] = b["closed"]
                            holder["val"] = b["val"]
                            holder["name"] = b["name"]
                            for t in b["attributes"]["text"]:
                                holder[t["name"]] = t["val"]

                        data.append(holder)

    return pd.DataFrame(data)




In [ ]:
df = data_to_dataframe()

# Analyse and filter Dataframe

## Locations covered by the dataset

Create interactive map for all the locations in dataset

In [ ]:
def create_gps_map(df, lat_col="latitude", lon_col="longitude", zoom_start=12, map_file="map.html"):
    """
    Create an interactive map from dataframe with latitude and longitude columns.
    """
    # Keep only unique coordinates
    unique_coordinates = df.drop_duplicates(subset=[lon_col, lat_col])

    if unique_coordinates.empty:
        raise ValueError("No valid latitude/longitude points found.")

    # Compute map center
    center_lat = unique_coordinates[lat_col].mean()
    center_lon = unique_coordinates[lon_col].mean()

    # Create the map
    m = folium.Map(location=[center_lat, center_lon], zoom_start=zoom_start)

    # Add markers
    for _, row in unique_coordinates.iterrows():
        folium.Marker(
            location=[row[lat_col], row[lon_col]],
            popup=f"Lat: {row[lat_col]}, Lon: {row[lon_col]}"
        ).add_to(m)

    # Save map
    m.save(map_file)
    print(f"Map saved as {map_file}")
    return m


In [ ]:

# Call function
create_gps_map(df, lat_col="latitude", lon_col="longitude", map_file="my_map.html")


## Duration that the dataset was recorded


In [ ]:
# earliest record
datetime.utcfromtimestamp(float(df.timestamp.min())).strftime("%Y-%m-%d %H:%M:%S")

In [ ]:
# latest record
datetime.utcfromtimestamp(float(df.timestamp.max())).strftime("%Y-%m-%d %H:%M:%S")

--> 6 days

## Hours at which data were collected

In [ ]:
# Prepare hours
df_hours = (
    df[["path", "timestamp"]]
    .drop_duplicates()
    .assign(hour=lambda d: pd.to_datetime(d["timestamp"].astype(float), unit="s").dt.hour)
)

# Aggregate counts per hour
hour_counts = df_hours.groupby("hour").size().reset_index(name="count")

# Plot as barplot
plt.figure(figsize=(15,5))
sns.barplot(data=hour_counts, x="hour", y="count", color="skyblue")
plt.title("Distribution of hours of day when images were generated")
plt.xlabel("Hour of day")
plt.ylabel("Count")
plt.show()


## Frames and seconds per folder (station)

In [ ]:
# Unique timestamps per dataset
df2_timestamps_unique = df.drop_duplicates(subset=["timestamp"], keep="first").copy()

# datetime
df2_timestamps_unique["timestamp"] = pd.to_datetime(df2_timestamps_unique["timestamp"], unit="s")

# Aggregate: count + time range
summary = (
    df2_timestamps_unique.groupby("dataset")["timestamp"]
    .agg(
        count="count",
        time_range_sec=lambda x: (x.max() - x.min()).total_seconds()
    )
    .reset_index()
    .sort_values("time_range_sec", ascending=False)  # sort by seconds
    .reset_index(drop=True)
)

print(summary)


For all stations we have either **10 frames (≈1 second)** or **100 frames (≈10 seconds)**,  
except for the following three cases:

- **12_vegetation_steady_12.1** — 98 frames (≈ 9.7 s)  
- **4_station_pedestrian_bridge_4.5** — 76 frames (≈ 7.4 s)  
- **21_station_wedel_21.3** — 40 frames (≈ 3.9 s)  


## Total number of photos:

In [ ]:
len(df.path.unique())

In [ ]:
df_temp = (
    df.groupby("type")
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)

# Plot
plt.figure(figsize=(10,5))
sns.barplot(data=df_temp, y="type", x="count", palette="viridis")
plt.title("Number of different images per class")
plt.xlabel("Number of images")
plt.ylabel("Class")
plt.show()

df_temp


--> Track is the second type with most labels

In [ ]:
# labels per sensor
df_temp = (
    df.groupby("sensor")
      .size()
      .reset_index(name="count")
      .sort_values("count", ascending=False)
)

# Plot
plt.figure(figsize=(10,5))
sns.barplot(data=df_temp, y="sensor", x="count", palette="mako")
plt.title("Number of labels per sensor")
plt.xlabel("Number of labels")
plt.ylabel("Sensor")
plt.show()

df_temp

--> rgb_highres_centre is the second sensor with msot labels

## Explore raw data

In [ ]:
def plot_time_series(df, value_col, time_col='timestamp', ylabel=None, title=None):
    """
    Plot a time series for a given value column against a timestamp column.
    """
    timestamps=df[time_col]
    values=df[value_col]

    # Convert timestamps to datetime objects
    datetime_timestamps=[datetime.utcfromtimestamp(float(ts)) for ts in timestamps]

    # Create the plot
    plt.figure(figsize=(10,5))
    plt.plot(datetime_timestamps, values, marker='o', linestyle='-', label=value_col)
    # plt.plot(timestamps, pitch_rate, marker='o', linestyle='-', label="Pitch Rate") #for original timestamps

    # Format x-axis
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S.%f')) #for datetime timestamps
    plt.xticks(rotation=45)

    # Labels & title
    plt.xlabel("Timestamp (UTC)")
    plt.ylabel(ylabel if ylabel else value_col)
    plt.title(title if title else f"{value_col} Over Time")
    plt.legend()
    plt.grid(True)
    # plt.xticks(timestamps[::10])  # Show every 10th timestamp--> for original timestamps
    # plt.xticks(rotation=45)  # Rotate x-axis labels for better readability --> for original timestamps
    plt.tight_layout()
    plt.show()



Example

100 frames (10 second)

In [ ]:
df2_1_calibration_12 = df[df['dataset'] == '1_calibration_1.2']

df2_1_calibration_12_unique_path = df2_1_calibration_12.drop_duplicates(subset=["path"], keep="first").copy()

df2_1_calibration_12_unique_path_rgbhigh = df2_1_calibration_12_unique_path[df2_1_calibration_12_unique_path['sensor'] == 'rgb_highres_center']  

#sensor can be anything else rather than 'rgb_highres_center' and it will give the same plot e.g. rgb_center:
is_consistent = df2_1_calibration_12.groupby(['sensor', 'timestamp'])['pitch_rate'].nunique().max() == 1
print("Pitch rate is the same for each unique sensor at each timestamp:", is_consistent)


In [ ]:
# cols=['pitch_rate','roll_rate','yaw_rate',
#       'lateral_acc','longitudinal_acc','vertical_acc']

cols=['pitch_rate','roll_rate']

for c in cols:
    plot_time_series(df2_1_calibration_12_unique_path_rgbhigh, c)


10 frames (1 second)

In [ ]:
df6_station_klein_flottbek = df[df['dataset'] == '6_station_klein_flottbek_6.1']

df6_station_klein_flottbek_unique_path = df6_station_klein_flottbek.drop_duplicates(subset=["path"], keep="first").copy()

df6_station_klein_flottbek_unique_path_rgbhigh = df6_station_klein_flottbek_unique_path[df6_station_klein_flottbek_unique_path['sensor'] == 'rgb_highres_center']  

plot_time_series(df6_station_klein_flottbek_unique_path_rgbhigh, 'pitch_rate')


--> Since records are only 1 or 10 seconds long--> will not be used in defect analysis

## Filter the dataframe --> we only care for tracks and rgb_highres_centre

In [ ]:
# for excluding lidar data:
df_track = df[(df["type"] == "track") & (df["label_type"] == "poly2d")].copy()  # same with: (df[(df["type"] == "track") & (df["sensor"] != "lidar")].equals(df[(df["type"] == "track") & (df["label_type"] == "poly2d")]))

#converts val values to (x,y) coordinates in a new column for the track data:
df_track["poly2d"] = df_track["val"].apply( 
    lambda x: np.array(np.reshape(x, (-1, 2)), dtype=np.int32)
)

df_track_rounded=df_track.copy() 

# round longitude and latitude to 6 decimal places (to consider them the same if they are very close):
df_track_rounded['longitude'] = df_track_rounded['longitude'].round(6)
df_track_rounded['latitude'] = df_track_rounded['latitude'].round(6)

# keep entries only from rgb camera placed in the center:
df_track_rgbchigh = df_track_rounded[df_track_rounded["sensor"] == "rgb_highres_center"].copy()

## Explore perspectives (different cameras mounted on car)

Function that prints each image with track lines

In [ ]:
def print_image(
    df, path, file, with_polyline=False, with_bounding_box=False, target=True,thresh=1000
):
    """
    Presents images with track lines(optional)
    """
    print(file)
    image = cv2.imread(path + file)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    window_name = "Image"
    if target:
        color = (0, 0, 255)
    else:
        color = (255, 0, 0)
    if not with_polyline:
        plt.imshow(image)
        plt.show()
        return
    for idx, row in df[df["path"] == file].iterrows():
        # if row["trackID"]=="-1" and row["railSide"]=="rightRail":
        #     color = (255, 0, 0)
        thickness = row.width // thresh 
        closed_val = row["closed"]
        pts = row["poly2d"].reshape((-1, 1, 2))
        image = cv2.polylines(image, [pts], closed_val, color, int(thickness))
        if with_bounding_box:
            w = max([i[0] for i in row.poly2d]) - min([i[0] for i in row.poly2d])
            h = max([i[1] for i in row.poly2d]) - min([i[1] for i in row.poly2d])
            x = min([i[0] for i in row.poly2d])
            y = min([i[1] for i in row.poly2d])
            image = cv2.rectangle(image, (x, y), (x + w, y + h), color, thickness // 2)
        # if target:
        #     color = (0, 0, 255)
        # else:
        #     color = (255, 0, 0)

    plt.imshow(image) 
    plt.show()
    # save_dir = os.getcwd() 
    # filename = os.path.basename(file)
    # output_path = os.path.join(save_dir, filename)
    # print(output_path)
    
    # cv2.imwrite(output_path, cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    # print(f"Saved annotated image to {output_path}")


In [ ]:
file = df_track[df_track["sensor"] == "ir_center"].path.unique()[0]
print_image(df_track, path=rail_data_path, file=file, with_polyline=True)

In [ ]:
file = df_track[df_track["sensor"] == "rgb_highres_right"].path.unique()[0]
print_image(df_track, path=rail_data_path, file=file, with_polyline=True)

In [ ]:
file = df_track[df_track["sensor"] == "rgb_highres_left"].path.unique()[0]
print_image(df_track, path=rail_data_path, file=file, with_polyline=True)

In [ ]:
file = df_track[df_track["sensor"] == "rgb_highres_center"].path.unique()[0]
print_image(df_track, path=rail_data_path, file=file, with_polyline=True)

In [ ]:
file = df_track[df_track["sensor"] == "rgb_center"].path.unique()[0]
print_image(df_track, path=rail_data_path, file=file, with_polyline=True)

--> In rgb_highres_center tracks are more visile and in better percpective so we keep only this

## Check brightness and entropy of images 

In [ ]:
# Unique paths
unique_paths = df_track_rgbchigh["path"].drop_duplicates().astype(str)

def brightness_entropy(image_path):
    """Return (brightness, entropy) for a single image path. 
       brightness in [0,255]; entropy in bits."""
    try:
        # Open and convert to grayscale
        with Image.open(image_path) as im:
            im = im.convert("L")  # grayscale
            arr = np.asarray(im, dtype=np.uint8)

        # Brightness: mean grayscale
        br = float(arr.mean())

        # Entropy: Shannon over 256-bin histogram
        hist = np.bincount(arr.ravel(), minlength=256).astype(np.float64)
        p = hist / hist.sum()
        # Avoid log(0) by masking zeros
        p = p[p > 0]
        ent = float(-(p * np.log2(p)).sum())

        return br, ent

    except Exception as e:
        # If unreadable/missing, return NaNs and print a warning
        print(f"Warning: {image_path} -> {e}")
        return np.nan, np.nan

# Compute metrics for each unique path
records = []
for rel_path in tqdm(unique_paths, desc="Processing images"):
    full_path = os.path.join(rail_data_path, rel_path)
    br, ent = brightness_entropy(full_path)
    records.append({"path": rel_path, "brigness": br, "entropy": ent})

# save f=df to CSV
out_df = pd.DataFrame(records, columns=["path", "brigness", "entropy"])
out_csv = "image_brightness_entropy.csv"
out_df.to_csv(out_csv, index=False)

print(f"Saved: {out_csv}")
out_df.head()

In [ ]:
df_metrics = pd.read_csv("image_brightness_entropy.csv")

In [ ]:
df_metrics

Highest and lowest brightness

In [ ]:
lowest_brightness = df_metrics.loc[df_metrics["brigness"].idxmin()]
print("Lowest brightness:\n", lowest_brightness, "\n")
print_image(df_track_rgbchigh, path=rail_data_path, file=lowest_brightness["path"], with_polyline=True)


highest_brightness = df_metrics.loc[df_metrics["brigness"].idxmax()]
print("Highest brightness:\n", highest_brightness, "\n")
print_image(df_track_rgbchigh, path=rail_data_path, file=highest_brightness["path"], with_polyline=True)

Highest and lowest entropy

In [ ]:
lowest_entropy    = df_metrics.loc[df_metrics["entropy"].idxmin()]
print("Lowest entropy:\n", lowest_entropy, "\n")
print_image(df_track_rgbchigh, path=rail_data_path, file=lowest_entropy["path"], with_polyline=True)


highest_entropy   = df_metrics.loc[df_metrics["entropy"].idxmax()]
print("Highest entropy:\n", highest_entropy, "\n")
print_image(df_track_rgbchigh, path=rail_data_path, file=highest_entropy["path"], with_polyline=True)


Exclude station 8 to see if the next lowest brightness and entropy is okay

In [ ]:
df_metrics2 = df_metrics[~df_metrics["path"].str.contains("8_station_altona_", na=False)]

lowest_brightness = df_metrics2.loc[df_metrics2["brigness"].idxmin()]
print("Lowest brightness:\n", lowest_brightness, "\n")
print_image(df_track_rgbchigh, path=rail_data_path, file=lowest_brightness["path"], with_polyline=True)

lowest_entropy    = df_metrics2.loc[df_metrics2["entropy"].idxmin()]
print("Lowest entropy:\n", lowest_entropy, "\n")
print_image(df_track_rgbchigh, path=rail_data_path, file=lowest_entropy["path"], with_polyline=True)


--> Tracks for station 8 are barely visible

## Aspect ratio (same for all entries since same camera-rgb_highres_center)

In [ ]:
df_track_rgbchigh['width'].unique(), df_track_rgbchigh['height'].unique()

In [ ]:
aspect_ratio = df_track_rgbchigh['width'].unique()/ df_track_rgbchigh['height'].unique()
aspect_ratio

## Number of rail track labels per image

As expected, rail tracks are mostly in pairs (left and right side). But, there are some images showing only one side of rail track.

In [ ]:
df_track_rgbchigh_1 = df_track_rgbchigh.copy()
df_track_rgbchigh_1["sensor_short"] = df_track_rgbchigh_1["sensor"].str.rsplit("_", n=1).str[-1]

# Count unique labels per path 
df_classes_count = (
    df_track_rgbchigh_1.groupby("path")["label_uid"]
    .nunique()
    .reset_index(name="classes_count")
)

# histogram
plt.figure(figsize=(15,5))
sns.histplot(df_classes_count["classes_count"], bins=range(1, df_classes_count["classes_count"].max()+2))
plt.title("Histogram of labels per image")
plt.xlabel("Number of labels")
plt.ylabel("Count")
plt.show()

## Correct wrong entries descovered

Some tracks are registered as left but those are right rails (discovered after manual checks). All the rows of railSide column are registered as left, but some of them are right:

In [ ]:
paths_of_interest = (
    '675_1631705207.600000008.png',
    '676_1631705207.700000031.png',
    '677_1631705207.800000022.png',
    '678_1631705207.900000013.png',
    '679_1631705208.000000005.png',
    '680_1631705208.100000028.png',
    '681_1631705208.200000019.png',
    '682_1631705208.300000010.png'
)

# Check:
df_track_rgbchigh[
    (df_track_rgbchigh['trackID'] == '0') &
    (df_track_rgbchigh['path'].str.endswith(paths_of_interest))
]


Correction and print:

In [ ]:
def fix_rail_side(row):
    if (
        row['tag'] == '7_approach_underground_station_7.2'
        and row['trackID'] == '0'
        and row['path'].endswith(paths_of_interest)
        and row['railSide'] == 'leftRail'
        and row['poly2d'][0][0] > 2000 # this is since the right rail is more to the right in the image (x>2000)
    ):
        return 'rightRail'
    return row['railSide']

df_track_rgbchigh['railSide'] = df_track_rgbchigh.apply(fix_rail_side, axis=1)

df_track_rgbchigh[
    (df_track_rgbchigh['trackID'] == '0') &
    (df_track_rgbchigh['path'].str.endswith(paths_of_interest))
]


Another wrong registration as left but is right:

In [ ]:
df_track_rgbchigh[
    (df_track_rgbchigh['trackID'] == '0') &
    (df_track_rgbchigh['path'].str.endswith('074_1631176103.500000016.png'))
]

Correction:

In [ ]:
mask = (
    (df_track_rgbchigh['path'] == '13_station_ohlsdorf_13.1/rgb_highres_center/074_1631176103.500000016.png')
    & (df_track_rgbchigh['trackID'] == '0')
    & (df_track_rgbchigh['poly2d'].str[0].str[0] >= 2500)
)

df_track_rgbchigh.loc[mask, 'railSide'] = 'rightRail'


df_track_rgbchigh[
    (df_track_rgbchigh['trackID'] == '0') &
    (df_track_rgbchigh['path'].str.endswith('074_1631176103.500000016.png'))
]

## Check whether all photos are different (i.e. maybe for some stations the camera was not moving)

Since images are frames from videos, there is low variation between the images - even less when the train is not moving.
This should be taken into account when generating train, val, and test sets (e.g. the accuracy of the model might not be realistic if model is trained, validated, and tested on similar images)

Function to plot photos from same station

In [ ]:
def images_on_grid(folder_path, n_images=7, figsize=(25, 25)):
    """
    Display the first n_images from a folder in a single-row grid.
    """
    sns.set(rc={"figure.figsize": figsize})

    files = glob(folder_path)
    if not files:
        raise ValueError(f"No images found in {folder_path}")

    # take up to n_images
    files = files[:n_images]

    # figure with n_images subplots in a row
    fig, axes = plt.subplots(nrows=1, ncols=len(files), figsize=figsize)

    # If only one image, axes is not iterable
    if len(files) == 1:
        axes = [axes]

    for ax, file in zip(axes, files):
        img = cv2.cvtColor(cv2.imread(file), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.axis("off")   #  hide ticks

    plt.subplots_adjust(wspace=0, hspace=0)
    plt.show()

All photos in the grid are the same-->car not moving

In [ ]:
images_on_grid("./rail_data/DB/4_station_pedestrian_bridge_4.1/rgb_highres_center/*")

Moving car:

In [ ]:
images_on_grid("./rail_data/DB/2_station_berliner_tor_2.1/rgb_highres_center/*")
images_on_grid("./rail_data/DB/7_approach_underground_station_7.1/rgb_highres_center/*")
images_on_grid("./rail_data/DB/9_station_ruebenkamp_9.1/rgb_highres_center/*")

If longitute and latitude is the same for the same camera (after the rounding to 6 decimls we performed) it means that the photo is the same. There are several entries because each line has different coordinates for track lines. This excludes from dataframe the photos that are taken from exactly the same place but keeps the important columns :

### Additional filtering

In [ ]:
#filter out photos that are taken from the same location (no motion)
df_track_rgbchigh_filtered = df_track_rgbchigh.drop_duplicates(subset=['longitude', 'latitude', 'tag', 'type', 'name', 
       'object_uid', 'sensor', 'height', 'width', 'dataset', 'label_type', 'railSide'], keep="first").copy()

In [ ]:
# check shape 
df_track_rgbchigh_filtered.shape #3946 26

## Occlusion

In [ ]:
# How visible each track is (no occlusion, low, medium, high)
df_track_rgbchigh_filtered['occlusion'].unique()

Check how an occlusion of 25-50 % is to decide if is good enough for keeping it

In [ ]:
df_1=df_track_rgbchigh_filtered[df_track_rgbchigh_filtered['occlusion'] == '25-50 %'].copy()
# for i in range(10):
#     print_image(df_1, rail_data_path, df_1['path'].unique()[i], with_polyline=True,thresh=100, target=False)

print_image(df_1, rail_data_path, df_1['path'].unique()[10], with_polyline=True, thresh=1000, target=False)

--> The track noted with the red line is barely visible--> defects cannot be detected from such tracks-->exclude them and keep only occlution of 0-25%:

### Additional filtering

In [ ]:
df_track_rgbchigh_filtered2 = df_track_rgbchigh_filtered[df_track_rgbchigh_filtered['occlusion'] == '0-25 %'].copy()

In [ ]:
df_track_rgbchigh_filtered2.shape #2929 26

In [ ]:
len(df_track_rgbchigh_filtered2['path'].unique()) #678 images in total kept

## Check how distance between a right and a left track changes with height of image 

There are certain points for each rail track label that all together create the polyline. The goal here is to make those points more with interpolation and then calculate all the distances (parallel to x axis) between a right and a left point 

Interpolation function:

In [ ]:
#interpolate x-coordinates at common_y 
def interpolate_poly(poly, new_y):
    """ takes as input a polyline (array of (x,y) coordinates) and a list of new y-coordinates (new_y)
        returns the interpolated x-coordinates at those new y-coordinates
    """
    poly = poly[np.argsort(poly[:, 1])] 
    x, y = poly[:, 0], poly[:, 1]

    y_unique, idx_unique = np.unique(y, return_index=True)
    x_unique = x[idx_unique]

    if len(y_unique) < 2:
        return None, None  # not enough points

    y_min, y_max = y_unique[0], y_unique[-1]
    safe_y = new_y[(new_y >= y_min) & (new_y <= y_max)]  # restrict to original range

    if len(safe_y) < 2:
        return None, None  # not enough valid points to compare

    f = interp1d(y_unique, x_unique, kind='linear', bounds_error=True) #Creates an interpolation function f that maps values from y_unique to x_unique using linear interpolation.
    x_interp = f(safe_y)

    return x_interp, safe_y
    

# Distinct colors for connector lines per pair
line_colors = [
    (255, 255, 255),  # white
    (255, 0, 255),    # magenta
    (0, 255, 255),    # cyan
    (255, 255, 0),    # yellow
    (0, 128, 255),    # orange-blue
    (128, 0, 255),    # purple
    (0, 255, 128),    # green-blue
    (255, 128, 0),    # orange
    (128, 255, 0),    # lime
    (0, 0, 255)       # blue
]



Plot how the distance between the points changes as y becomes smaller (further in the image), next to the original image. 
In the original image:
- Original given points are shown as **large dots**.
- Interpolated points are shown as **smaller dots**.
- **Lines** between matched points are drawn to illustrate how distances are computed.

This side-by-side view confirms that the distance calculation behaves as expected as *y* becomes smaller.

In [ ]:

def plot_railwidthVsHeight_width_image(df,number_test):
    """ Plots rail width vs. image height for matched left and right rails
        Also shows the original image with polylines and interpolated points (small) with original points (bigger)
        number_test: number of random images to plot
    """
    
    
    df = df.dropna(subset=['poly2d'])

    unique_paths = df['path'].unique()[random.sample(range(100), number_test)] #choose tandomly some photos to plot to check if code works proberly
    
    jj=0
    for selected_path in tqdm(unique_paths, desc="Processing images"):
        print(f"\n Now processing: {selected_path}")
        df_img = df[df['path'] == selected_path].copy()
        df_img = df_img.dropna(subset=['poly2d'])
        df_left = df_img[df_img['railSide'] == 'leftRail'].reset_index(drop=True)
        df_right = df_img[df_img['railSide'] == 'rightRail'].reset_index(drop=True)

 
        img_path =  os.path.join(rail_data_path, selected_path) # Load 
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) #convert colors
        H, W = img.shape[:2] #get dimensions
        common_y = np.linspace(0, H , 100) #define how many points between 0 and H you need


        fig, axs = plt.subplots(1, 2, figsize=(14, 6))

        # Match right to left track using trackID column
        matches = []

        df_left['trackID'] = df_left['trackID'].astype(str)
        df_right['trackID'] = df_right['trackID'].astype(str)
        track_ids = set(df_right['trackID']).intersection(df_left['trackID']) #set removes duplicates and intersection finds common elements

        for track_id in track_ids:
            poly_r_list = df_right[df_right['trackID'] == track_id]['poly2d'].values
            poly_l_list = df_left[df_left['trackID'] == track_id]['poly2d'].values

            if len(poly_r_list) == 0 or len(poly_l_list) == 0:
                continue

            poly_r = poly_r_list[0]
            poly_l = poly_l_list[0]

            matches.append((poly_r, poly_l))  # one pair per trackID

        if len(matches) == 0:
            print(f"No valid rail pairs in image: {selected_path}")
            continue
            
        #Plot and compare interpolated values for each matched pair
        for i, (poly_r_list, poly_l_list) in enumerate(matches): #to have Legends (label=f'Pair {i}') and Color cycling for lines 

            # Interpolate each polyline over shared y-range
            x_r_interp, safe_y_r = interpolate_poly(poly_r_list, common_y)
            x_l_interp, safe_y_l = interpolate_poly(poly_l_list, common_y)

            if x_r_interp is None or x_l_interp is None:
                continue

            safe_y = np.intersect1d(safe_y_r, safe_y_l) #keeps only common elements
            if len(safe_y) < 2:
                continue

            x_r_final = interp1d(safe_y_r, x_r_interp)(safe_y)
            x_l_final = interp1d(safe_y_l, x_l_interp)(safe_y)

            # Plot 1 rail width vs. height
            axs[0].set_ylim(0, H)
            axs[0].invert_yaxis()
            axs[0].plot(np.abs(x_r_final - x_l_final), safe_y, marker='o', label=f'Pair {i}')
            axs[0].set_ylabel("Image height (y)")
            axs[0].set_xlabel("Rail width (pixels)")
            axs[0].set_title("Rail width vs. height")
            axs[0].grid(True)

            # Draw on image bigger dots for original raw polyline points for reference 
            for pt in poly_r_list:
                cv2.circle(img_rgb, (int(pt[0]), int(pt[1])), 30, (255, 0, 0), -1)
            for pt in poly_l_list:
                cv2.circle(img_rgb, (int(pt[0]), int(pt[1])), 30, (0, 255, 0), -1)

            # Draw interpolated points
            for xi_l, xi_r, yi in zip(x_l_final, x_r_final, safe_y):
                pt_l = (int(np.clip(xi_l, 0, W-1)), int(np.clip(yi, 0, H-1)))
                pt_r = (int(np.clip(xi_r, 0, W-1)), int(np.clip(yi, 0, H-1)))
                cv2.circle(img_rgb, pt_l, 15, (0, 255, 0), -1)   # left: green
                cv2.circle(img_rgb, pt_r, 15, (255, 0, 0), -1)   # right: red
                cv2.line(img_rgb, pt_l, pt_r, line_colors[i % len(line_colors)], 8)  # white connector

            axs[0].legend()


        # Image display with axes
        axs[1].set_ylim(0, H)
        axs[1].invert_yaxis()
        axs[1].imshow(img_rgb)
        axs[1].set_xlabel("x (pixel)")
        axs[1].set_ylabel("y (pixel)")
        axs[1].set_title("Image with rail polylines and crop zone")

        plt.tight_layout()
        if jj==4:
            fig_single, ax_single = plt.subplots(figsize=(8, 6))
    
            # Copy the image from axs[1] to the new ax
            ax_single.imshow(img_rgb)
            ax_single.set_aspect('auto')
            ax_single.set_title("Image with rail polylines and crop zone",  fontsize=18)
            ax_single.set_xlabel("x (pixel)", fontsize=17)
            ax_single.set_ylabel("y (pixel)", fontsize=17)
            ax_single.set_ylim(0, H)
            ax_single.invert_yaxis()
            ax_single.tick_params(axis='both', labelsize=16)


            plt.tight_layout()
            # fig_single.savefig("third_ax1_only.pdf", format="pdf", 
            #  bbox_inches='tight',   
            #  pad_inches=0.05)
            plt.close(fig_single)  # close extra figure after saving
            
        plt.show()
        jj=jj+1


In [ ]:
plot_railwidthVsHeight_width_image(df_track_rgbchigh,5)

--> So it works properly

### Plot rail track distance vs height taking into account all images

In [ ]:
def analyze_cropping_range(df,num_rail_points=100):
   
    
    df = df.dropna(subset=['poly2d'])

    unique_paths = df['path'].unique()

    all_y_vals = []
    all_distances = []

    for selected_path in tqdm(unique_paths, desc="Processing images"):
        df_img = df[df['path'] == selected_path].copy()
        df_img = df_img.dropna(subset=['poly2d'])
        df_left = df_img[df_img['railSide'] == 'leftRail'].reset_index(drop=True)
        df_right = df_img[df_img['railSide'] == 'rightRail'].reset_index(drop=True)

        img_path = os.path.join(rail_data_path, selected_path)
        img = cv2.imread(img_path)
        if img is None:
            raise FileNotFoundError(f"Image not found: {img_path}")
        # img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) #convert colors
        H, W = img.shape[:2]
        common_y = np.linspace(0, H, num_rail_points)

        # fig, axs = plt.subplots(1, 2, figsize=(14, 6))

        # # Match right to left track using trackID column
        # matches = []

        # TrackID-based matching
        df_left['trackID'] = df_left['trackID'].astype(str)
        df_right['trackID'] = df_right['trackID'].astype(str)
        track_ids = set(df_right['trackID']).intersection(df_left['trackID'])

        for track_id in track_ids:
            poly_r_list = df_right[df_right['trackID'] == track_id]['poly2d'].values
            poly_l_list = df_left[df_left['trackID'] == track_id]['poly2d'].values

            if len(poly_r_list) == 0 or len(poly_l_list) == 0:
                continue

            poly_r = poly_r_list[0]
            poly_l = poly_l_list[0]

            x_r_interp, safe_y_r = interpolate_poly(poly_r, common_y)
            x_l_interp, safe_y_l = interpolate_poly(poly_l, common_y)

            if x_r_interp is None or x_l_interp is None:
                continue

            safe_y = np.intersect1d(safe_y_r, safe_y_l)
            if len(safe_y) < 2:
                continue

            x_r_final = interp1d(safe_y_r, x_r_interp)(safe_y)
            x_l_final = interp1d(safe_y_l, x_l_interp)(safe_y)
            distances = np.abs(x_r_final - x_l_final)

            all_y_vals.extend(safe_y)
            all_distances.extend(distances)

    df_summary = pd.DataFrame({'y': all_y_vals, 'distance': all_distances})

    # Bin by height
    df_summary['y_bin'] = pd.cut(df_summary['y'], bins=50) #devide into bins the different heights
    grouped = df_summary.groupby('y_bin').agg({'distance': ['mean', 'median', 'std'], 'y': 'mean'}).reset_index()
    grouped.columns = ['y_bin', 'distance_mean', 'distance_median', 'distance_std', 'y']

    # Plot median rail distance vs height
    plt.figure(figsize=(8, 6))
    plt.plot(grouped['distance_median'], grouped['y'], label='Median distance')
    plt.fill_betweenx(grouped['y'],
                      grouped['distance_median'] - grouped['distance_std'],
                      grouped['distance_median'] + grouped['distance_std'],
                      alpha=0.3, label='±1 STD')
    plt.gca().invert_yaxis()
    plt.xlabel("Distance between right and left rail (pixels)", fontsize=17)
    plt.ylabel("Image height (y)", fontsize=17)
    plt.title("Rail distance vs. vertical image position (aggregated)",  fontsize=18)
    plt.grid(True)
    plt.tick_params(axis='both', labelsize=16)
    plt.legend()
    plt.tight_layout()
    # plt.savefig("rail_distance_vs_height.pdf", format='pdf',
    #          bbox_inches='tight',   
    #          pad_inches=0.05)
    plt.show()


    # Get the clean values
    x_vals = grouped['distance_median'].values
    y_vals = grouped['y'].values

    # Remove any NaNs(avoid errors in interpolation)
    valid = np.isfinite(x_vals) & np.isfinite(y_vals)
    x_vals = x_vals[valid]
    y_vals = y_vals[valid]

    #Create interpolation function: y = f(distance)
    f_y_from_distance = interp1d(x_vals, y_vals, kind='linear', bounds_error=False, fill_value=np.nan)

    #Get y values for desired distances
    return f_y_from_distance



In [ ]:
f_y_from_distance1 = analyze_cropping_range(df_track_rgbchigh_filtered2,num_rail_points=100) 

--> Since the relationship between rail distance and image height is approximately linear with only slight variations, we selected two representative y positions at 500 and 800. These correspond to regions of the image that are relatively close (and therefore clearer), while still maintaining comparable track widths. We exclude sections above 800 and below 500 to avoid very small or very large apparent track distances.

In [ ]:
f_y_from_distance1(800),f_y_from_distance1(500) 

If we have rail track patches of 64 pixels and each patch covers 50 % of its prior patch, this will result in 21 patches per rail track: 

In [ ]:
(2231-1548.7)/64*2

## Measure widths of rail tracks (can be skipped) 

Here the goal is to measure the width of each rail track as appeared in each image. Therefore, we measure it manually for one photo per station , since for the rest photos it is the same rail track 

In [ ]:
class ImageDistanceCalculator:
    def __init__(self):
        self.points = []
        self.image = None
        self.image_copy = None
        self.image_name = ""
        self.distances = []
        self.results = []    
    
    def reset(self):
        """Reset points for the next image"""
        self.points = []
    
    def get_coordinates(self, event, x, y, flags, param):

        """Mouse callback function to get coordinates and calculate distance"""
        if event == cv2.EVENT_LBUTTONDOWN:
            # Add the point to our list (only if we have less than 2)
            if len(self.points) < 2:
                self.points.append((x, y))
                print(f"[{self.image_name}] Point {len(self.points)} selected: ({x}, {y})")
                
                # Draw a circle at the clicked point
                cv2.circle(self.image_copy, (x, y), 5, (0, 0, 255), -1)
                cv2.imshow(self.image_name, self.image_copy)
                
                # If we have two points, calculate the distance
                if len(self.points) == 2:
                    p1, p2 = self.points
                    # Calculate Euclidean distance
                    distance = math.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)
                    print(f"[{self.image_name}] Pixel distance between the two points: {distance:.2f} pixels")
                    self.distances.append(distance)

                    self.results.append((self.image_path, distance))
                    
                    # Draw a line between the two points
                    cv2.line(self.image_copy, p1, p2, (0, 255, 0), 2)
                    cv2.imshow(self.image_name, self.image_copy)

                    
    def process_image(self, image_path):
        """Process a single image"""
        try:

            self.image_path = image_path  
            # Extract just the filename from the path
            self.image_name = os.path.basename(image_path)
            
            # Read the image
            self.image = cv2.imread(image_path)
            
            if self.image is None:
                print(f"Error: Could not read the image at {image_path}")
                return False
            
            # Create a copy for drawing
            self.image_copy = self.image.copy()
            
            # Create a window and set the mouse callback
            # cv2.namedWindow(self.image_name)

            cv2.namedWindow(self.image_name, cv2.WINDOW_NORMAL)
            cv2.setWindowProperty(
            self.image_name,
            cv2.WND_PROP_FULLSCREEN,
            cv2.WINDOW_FULLSCREEN)
            cv2.setMouseCallback(self.image_name, self.get_coordinates)
            
            # Display the image
            cv2.imshow(self.image_name, self.image)
            print(f"\nProcessing image: {self.image_name}")
            print("Click on two points to measure the pixel distance. Press any key when done to move to the next image.")
            
            # Wait for a key press and then close the window
            cv2.waitKey(0)
            cv2.destroyWindow(self.image_name)
            
            # Reset points for the next image
            self.reset()
            return True
            
        except Exception as e:
            print(f"An error occurred while processing {image_path}: {e}")
            return False

def main():
    # List of image paths
    image_paths = [
    os.path.join(rail_data_path, file)
    for file in df_folders.path.unique()
]
    unique_stations = {}
    filtered_paths = []

    for path in image_paths:
        # Extract station identifier, e.g. '14_signals_station_14.1'
        station = os.path.normpath(path).split(os.sep)[2][:-2]
        if station not in unique_stations:
            unique_stations[station] = path
            filtered_paths.append(path)
    
    image_paths = filtered_paths
    # Create the calculator object
    calculator = ImageDistanceCalculator()
    
    # Process each image
    for i, image_path in enumerate(image_paths, 1):
        print(f"\nImage {i} of {len(image_paths)}, path: {image_path}")
        success = calculator.process_image(image_path)
        if not success:
            print(f"Skipping to next image...")
    
    print("\nAll images processed. Exiting...")
    return calculator.results



In [ ]:
results = main()   

df_folders = df_track_rgbchigh.drop_duplicates(subset='tag', keep='first')
dists   = np.array([r[1] for r in results], dtype=float)
mean_dist = np.mean(dists)
std_dist  = np.std(dists, ddof=1)  # sample std

print("\n=== Summary ===")
print(f"Measured {len(dists)} distances")
print(f"Mean    = {mean_dist:.2f} px")
print(f"Std dev = {std_dist:.2f} px")

In [ ]:
sorted_asc  = sorted(results, key=lambda x: x[1])

In [ ]:
sorted_asc

We choose the median: 

In [ ]:
np.median(dists) #35

## Check where on each track the polyline points are loocated 


The reported points of the tracks are located on the outer side (print one from each file):

In [ ]:
df_folders = df_track_rgbchigh.drop_duplicates(subset='tag', keep='first')

files =  df_folders.path.unique() #df_track_rgbchigh[df_track_rgbchigh["tag"] == "12_vegetation_steady_12.1"].path.unique()[0]
for file in files:
    print_image(df_track_rgbchigh, path=rail_data_path, file=file, with_polyline=True)

### Incompatibility observed: 

Even though in almost all the photos the reported track points are located on their outer edges, there are some labels reported on the insight edges. This is taken into account later, in function get_masked_image_shifted(), where the masks for tracks are created.

In [ ]:
df_test1 = df_track_rgbchigh

# df_test1 = df_test1[df_test1['trackID'] == '-1']
df_test1 = df_test1[df_test1['trackID'].isin(['-1', '0'])]
print_image(df_test1, path=rail_data_path, file='14_signals_station_14.1/rgb_highres_center/093_1631449460.400000020.png',thresh=500, with_polyline=True)
# print_image(df_test1, path=rail_data_path, file='14_signals_station_14.1/rgb_highres_center/094_1631449460.500000011.png',thresh=1000, with_polyline=True)
# print_image(df_test1, path=rail_data_path, file='14_signals_station_14.1/rgb_highres_center/095_1631449460.600000003.png',thresh=1000, with_polyline=True)
# print_image(df_test1, path=rail_data_path, file='14_signals_station_14.1/rgb_highres_center/096_1631449460.700000026.png',thresh=1000, with_polyline=True)

df_test4 = df_track_rgbchigh
# df_test4 = df_test4[df_test4['trackID'] == '-1']
df_test4 = df_test4[df_test4['trackID'].isin(['-1', '0'])]


print_image(df_test4, rail_data_path,'7_approach_underground_station_7.1/rgb_highres_center/151_1631705118.200000025.png', with_polyline=True,thresh=500)
# print_image(df_test4, rail_data_path,'7_approach_underground_station_7.1/rgb_highres_center/153_1631705118.400000015.png', with_polyline=True,thresh=1000, target=False)



### Move the rail track lines to be in the middle (to take same pixels from poth sides for the patch crop)

Function to check whether it is moved properly:

In [ ]:
def print_image_shifted(
    df, path, file,
    with_polyline=False,
    shift_min=1, shift_max=17.5,
    thresh=100,
    target=True
):
    #load + convert for matplotlib
    img_bgr = cv2.imread(os.path.join(path, file))
    if img_bgr is None:
        raise FileNotFoundError(file)
    img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    #make two independent copies
    orig_img    = img.copy()
    shifted_img = img.copy()
    h, w = img.shape[:2]

    if not with_polyline:
        plt.imshow(img); plt.axis("off"); plt.show()
        return

    color = (0,255,0) if target else (255,0,0)

    #draw each rail‐polyline
    for _, row in df[df["path"] == file].iterrows():
        pts = row["poly2d"].reshape(-1,2).astype(np.float32)
        if pts[0,1] < pts[-1,1]:
            pts = pts[::-1]
    
        closed = bool(row["closed"])
        side   = row.get("railSide")
        if side is None:
            continue

        # a sane thickness >= 1 px
        thickness = max(int(row.width / thresh //10 ), 1)

        # draw the *original* polyline on orig_img
        pts_int = pts.astype(np.int32).reshape(-1,1,2)
        cv2.polylines(orig_img, [pts_int], closed, color, thickness)

        # compute shifted segments
        segments = []
        for i in range(len(pts)-1):
            p1, p2 = pts[i], pts[i+1]
            d = p2 - p1
            norm = np.linalg.norm(d)
            if norm == 0:
                continue
            u = d / norm
            n = np.array([-u[1], u[0]])
            yavg = (p1[1] + p2[1]) / 2
            rel  = yavg / h
            shift_dist = shift_min + (shift_max - shift_min) * rel
            sign = 1 if side == "leftRail" else -1
            shift = n * shift_dist * sign
            segments.append((p1 + shift, p2 + shift))

        if not segments:
            continue

        # flatten into a single polyline
        all_pts = np.vstack([
            np.array(seg, dtype=np.int32).reshape(2,1,2)
            for seg in segments
        ]).reshape(-1,1,2)

        # draw shifted on shifted_img
        cv2.polylines(shifted_img, [all_pts], closed, color, thickness)

    # show side by side
    fig, (ax1,ax2) = plt.subplots(1,2, figsize=(12,5))
    ax1.imshow(orig_img);    ax1.set_title("Original"); ax1.axis("on")
    ax2.imshow(shifted_img); ax2.set_title("Shifted");  ax2.axis("on")
    plt.tight_layout()
    plt.show()


In [ ]:
df_folders = df_track_rgbchigh.drop_duplicates(subset='tag', keep='first')

files =  df_folders.path.unique() #df_track_rgbchigh[df_track_rgbchigh["tag"] == "12_vegetation_steady_12.1"].path.unique()[0]
for file in files:
    print(file)
    print_image_shifted(df_track_rgbchigh, path=rail_data_path, file=file, with_polyline=True)

--> From the above examples we can see that the rail track polyline is correctly moved to the middle

# Create masks (keep only the rail track part and everything else be set to black) and crop the patches 

## All the necessary functions:

In [ ]:
def display_original_masked_mask(original, masked, mask):
    """
    Display original image, masked image, and mask side by side
    """
    plt.figure(figsize=(18,6))
    
    plt.subplot(1, 3, 1)
    plt.imshow(original)
    plt.title('Original Image')
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(masked)
    plt.title('Masked Image (only rails)')
    plt.axis('off')
    
    plt.subplot(1, 3, 3)
    plt.imshow(mask, cmap='gray')  # mask is grayscale
    plt.title('Mask (binary)')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

def get_masked_image_shifted(df, path, file, shift_max=17.5, shift_min=1, min_thickness=1, max_thickness=35, scale=1):
    #code that was tested with print_image shifted for shifting the rail track polyline is used here
    #Also the wrong railSide labelling discovered in some images is fixed here
    #max thickness is chosen based on the median rail distance at the bottom of the image calculated with class ImageDistanceCalculator above
    #shif_max is the max_thickness/2
    #min_thickness is chosen to be 1 since rail trucks are very thin at the top of the image
    #scale is 1 since edges are smooth and no supersampling is needed
    """
    Create a *soft* rail mask using {scale}× supersampling to not have jagged borders:
      1) draw on a high-res mask (scale×),
      2) Pre-calculate smoothed thickness values for all points
      3) Draw line segments with consistent thickness per segment
      4) downsample with INTER_AREA,
      5) keep mask soft (0..255) until patch extraction.
    """

    image = cv2.imread(path + file)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    original_image = image.copy()
    height, width = image.shape[:2]
    
    # Supersampled canvas
    Hs, Ws = height * scale, width * scale
    big_mask = np.zeros((Hs, Ws), dtype=np.uint8)
    
    for idx, row in df[df["path"] == file].iterrows():
        pts = row["poly2d"].reshape((-1, 2)).astype(np.float32)
        if pts[0, 1] < pts[-1, 1]:
            pts = pts[::-1]
        
        side = row.get("railSide")
        if side is None:
            continue
        
        if len(pts) < 2:
            continue
        
        # Pre-calculate thickness for each point based on y-position
        point_thickness = []
        for pt in pts:
            relative_position = pt[1] / height
            thickness_f = (min_thickness + (max_thickness - min_thickness) * relative_position) * scale
            point_thickness.append(thickness_f)
        
        # Apply smoothing to thickness values to eliminate abrupt changes
        if len(point_thickness) > 4:
            # Convert to numpy for easier manipulation
            thickness_array = np.array(point_thickness, dtype=np.float32)
            
            # Apply Gaussian smoothing with a larger kernel
            kernel_size = min(len(point_thickness) // 2, 5) if len(point_thickness) > 10 else 3
            if kernel_size % 2 == 0:
                kernel_size += 1
            
            # Use Gaussian blur for smoother transitions
            thickness_array = thickness_array.reshape(1, -1)
            smoothed = cv2.GaussianBlur(thickness_array, (kernel_size, 1), sigmaX=2.0)
            smooth_thickness = smoothed.flatten().tolist()
        else:
            smooth_thickness = point_thickness
        
        # Ensure minimum thickness
        smooth_thickness = [max(1, t) for t in smooth_thickness]
        
        # Calculate shifted points and draw segments
        for i in range(len(pts) - 1):
            p1, p2 = pts[i], pts[i + 1]
            d = p2 - p1
            norm = np.linalg.norm(d)
            if norm == 0:
                continue  # skip duplicate points
            
            u = d / norm
            n = np.array([-u[1], u[0]])
            
            # Calculate shift distances for both points
            y1, y2 = p1[1], p2[1]
            rel_pos1 = y1 / height
            rel_pos2 = y2 / height
            shift_dist1 = shift_min + (shift_max - shift_min) * rel_pos1
            shift_dist2 = shift_min + (shift_max - shift_min) * rel_pos2
            
            sign = 1 if side == "leftRail" else -1
            
            #Exception because the points of this rail track are differently defined from other images (not on out side)
            if file.split('/')[0] == '14_signals_station_14.1' and file.endswith((
                '093_1631449460.400000020.png', '094_1631449460.500000011.png',
                '095_1631449460.600000003.png', '096_1631449460.700000026.png'
            )) and row.get("railSide") == "rightRail" and row.get("trackID") == '-1':
                sign = 1
            elif file.split('/')[0] == '7_approach_underground_station_7.1' and file.endswith((
                '151_1631705118.200000025.png', '153_1631705118.400000015.png'
            )) and row.get("railSide") == "rightRail" and row.get("trackID") == '-1':
                sign = 1
            
            # Calculate shifted points
            shift1 = n * shift_dist1 * sign
            shift2 = n * shift_dist2 * sign
            shifted_p1 = p1 + shift1
            shifted_p2 = p2 + shift2
            
            # Scale coordinates and round
            x1, y1 = np.round(shifted_p1 * scale).astype(int)
            x2, y2 = np.round(shifted_p2 * scale).astype(int)
            
            # Use the smoothed thickness at the START point of the segment
            # ensures consistency - each segment uses one thickness value
            thickness = max(1, int(round(smooth_thickness[i])))
            
            # Draw the line segment with consistent thickness
            cv2.line(big_mask, (x1, y1), (x2, y2), 255, thickness=thickness, lineType=cv2.LINE_AA)
    
    if scale > 1:
        # Downsample with area averaging (keeps edges straight)
        soft_mask = cv2.resize(big_mask, (width, height), interpolation=cv2.INTER_AREA)
        # Keep mask soft
        masked_image = (original_image * (soft_mask[..., None].astype(np.float32) / 255.0)).astype(np.uint8)
    elif scale == 1:
        # Mild blur to soften single-resolution aliasing
        soft_mask = cv2.GaussianBlur(big_mask, (0, 0), sigmaX=0.8, sigmaY=0.8, borderType=cv2.BORDER_REPLICATE)
        masked_image = (original_image * (soft_mask[..., None].astype(np.float32) / 255.0)).astype(np.uint8)
    else:
        raise ValueError("scale must be ≥ 1")
    
    return original_image, masked_image, soft_mask





def extract_patches_from_interpolated_polyline(original_image, masked_image, mask, poly2d, patch_width=32, vertical_crop_limits=(500, 2000), horizontal_crop_limits=(750, 3250), num_points=100, specific_y=None):

    """
    Extract rectangular patches at evenly spaced points along the polyline,
    oriented along the direction of the track.

    Args:
        image: RGB image.
        mask: Binary mask (same size as image) with rail lines drawn in white (255).
        poly2d: Sparse polyline.
        patch_width: Width of patch (in pixels).
        vertical_crop_limits: Vertical limits for patch extraction.
        num_points: Number of points to interpolate along the polyline.
        specific_y: If provided, extract patch only at this y-coordinate (for sequential dataset)

    Returns:
        patches: List of valid patches.
        positions: List of (x, y, angle) positions and angles.
    """

    h, w, _ = original_image.shape
    original_patches, masked_patches, positions = [], [], []

    common_y = np.linspace(vertical_crop_limits[0], vertical_crop_limits[1], num_points)

    if specific_y is not None:
        common_y = np.linspace(specific_y - 5, specific_y + 5, num=10)  # small window for interpolation
    else:
        common_y = np.linspace(vertical_crop_limits[0], vertical_crop_limits[1], num_points)
    
    
    x_interp, safe_y = interpolate_poly(poly2d, common_y)
    if x_interp is None or safe_y is None:
        return [], [], []  # handle case when interpolation fails
    


    # Find the index closest to specific_y
    if specific_y is not None:
        idx_closest = np.argmin(np.abs(safe_y - specific_y))

        # Optionally compute angle with nearest neighbor if possible
        if 0 < idx_closest < len(safe_y) - 1:
            dx = x_interp[idx_closest + 1] - x_interp[idx_closest - 1]
            dy = safe_y[idx_closest + 1] - safe_y[idx_closest - 1]
            angle = round(np.degrees(np.arctan2(dy, dx)), 3)
        else:
            angle = 0  # fallback angle
        interpolated_pts = [(x_interp[idx_closest], safe_y[idx_closest], angle)]

    else:
        # Calculate track angles between consecutive points
        angles = []
        for i in range(len(x_interp) - 1):
            dx = x_interp[i+1] - x_interp[i]
            dy = safe_y[i+1] - safe_y[i]
            angle = round(np.degrees(np.arctan2(dy, dx)), 3)
            angles.append(angle)
        
        # Add the last angle (use the previous one)
        if angles:
            angles.append(angles[-1])
        else:
            return [], [], []  # Not enough points to calculate angles
        
        interpolated_pts = list(zip(x_interp, safe_y, angles))
    
    min_y = vertical_crop_limits[0]
    max_y = vertical_crop_limits[1]

    
    for i, (x_center, y_center, angle) in enumerate(interpolated_pts):
        if not (min_y <= y_center <= max_y):
            continue  # Skip if the point is not in the middle vertical section
        if not (horizontal_crop_limits[0] <= x_center <= horizontal_crop_limits[1]):
            continue 

        # Round the coordinates
        x_center = int(round(x_center))
        y_center = int(round(y_center))
        
        # Create a rotation matrix
        M = cv2.getRotationMatrix2D((patch_width//2, patch_width//2), angle-90, 1.0)
        
        # Extract a larger patch to ensure we have enough pixels after rotation
        safe_margin = int(patch_width * 1.5)
        x1 = max(0, x_center - safe_margin // 2)
        x2 = min(w, x_center + safe_margin // 2)
        y1 = max(0, y_center - safe_margin // 2)
        y2 = min(h, y_center + safe_margin // 2)
        
        # Skip if we can't extract a full patch
        if x1 == 0 or y1 == 0 or x2 == w or y2 == h:
            continue
        
        # Extract the larger patch
        large_patch = masked_image[y1:y2, x1:x2]
        
        # Skip if the patch is too small
        if large_patch.shape[0] < safe_margin or large_patch.shape[1] < safe_margin:
            continue
        
        # Resize to ensure consistent dimensions
        large_patch = cv2.resize(large_patch, (safe_margin, safe_margin))
        
        # Rotate the patch
        rotated_patch = cv2.warpAffine(large_patch, M, (safe_margin, safe_margin))
        
        # Crop the center to get the final patch size
        start = safe_margin//2 - patch_width//2
        end = start + patch_width
        final_patch = rotated_patch[start:end, start:end]
        
        # Skip if the final patch doesn't have the correct dimensions
        if final_patch.shape[0] != patch_width or final_patch.shape[1] != patch_width:
            continue
        
        masked_patches.append(final_patch)
        positions.append((x_center, y_center, angle))  # Store position and angle


        # Extract the larger patch
        large_patch2 = original_image[y1:y2, x1:x2]
        
        # Skip if the patch is too small
        if large_patch2.shape[0] < safe_margin or large_patch2.shape[1] < safe_margin:
            continue
        
        # Resize to ensure consistent dimensions
        large_patch2 = cv2.resize(large_patch2, (safe_margin, safe_margin))
        
        # Rotate the patch
        rotated_patch2 = cv2.warpAffine(large_patch2, M, (safe_margin, safe_margin))
        
        # Crop the center to get the final patch size
        start = safe_margin//2 - patch_width//2
        end = start + patch_width
        final_patch2 = rotated_patch2[start:end, start:end]
        
        # Skip if the final patch doesn't have the correct dimensions
        if final_patch2.shape[0] != patch_width or final_patch2.shape[1] != patch_width:
            continue
        
        original_patches.append(final_patch2)


    return original_patches, masked_patches, positions




def process_rail_patches(df, path, file, patch_size=32, thresh=100, vertical_crop_limits=(500, 2000), horizontal_crop_limits=(750, 3250), idx=0, num_points=100,print_thresh=10,specific_y=None, preprocesspatches=True):
    """
    Process an image to extract rail patches aligned with track direction and visualize them.
    """
    
    temp_df = df.copy()
    mask_df = (temp_df["path"] == file)
    image_path = os.path.join(path, file)
    original_image = cv2.imread(image_path)
    if original_image is None:
        print(f"Error: Could not read image from {image_path}")
        return

    # Convert BGR -> RGB for patch extraction 
    original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
    
    # Get points for all polylines of image
    polylines = [row["poly2d"] for _, row in temp_df[mask_df].iterrows()]
    if not polylines:
        print("No polylines found for this image.")
        return
    
    # Create mask (as in original code)
    mask = np.zeros(original_image.shape[:2], dtype=np.uint8)
    
    # Extract patches with orientation
    original_rail_patches = []
    masked_rail_patches = []
    patch_positions = []
    patch_track_ids = []


    ######mask##################
    original_im, masked_image, maskk = get_masked_image_shifted(df, path=rail_data_path, file=file)

    if idx % print_thresh == 0 and preprocesspatches:
        display_original_masked_mask(original_im, masked_image, maskk)
    ############################


    pidx = 1
    patch_sides = []

    for (_, row), poly in zip(temp_df[mask_df].iterrows(), polylines):
        rail_side = row.get("railSide", "unknown") 
        track_id = row.get("trackID", "unknown")

        if row.get("tag", "unknown") == '1_calibration_1.2' and specific_y is not None :
            horizontal_crop_limits=(750, 4000)
        original_patches_i, masked_patches_i, positions_i = extract_patches_from_interpolated_polyline(
            original_image,
            masked_image,
            mask,
            poly,
            patch_width=patch_size,
            vertical_crop_limits=vertical_crop_limits,
            horizontal_crop_limits=horizontal_crop_limits,
            num_points=num_points,
            specific_y=specific_y
        )
        original_rail_patches.extend(original_patches_i)
        masked_rail_patches.extend(masked_patches_i)
        patch_positions.extend(positions_i)

        rail_sides = [rail_side] * len(masked_patches_i)
        patch_sides.extend(rail_sides)

        # replicate the track_id for each patch
        track_ids_i = [track_id] * len(masked_patches_i)      
        patch_track_ids.extend(track_ids_i)    
        
        if idx % print_thresh == 0:
            print(f"Polyline {pidx}: {len(masked_patches_i)} patches kept")
        pidx += 1
    
    # Save all patches
    if specific_y is not None:
        base_patches_dir = Sequential_patches_dir
    else:
        base_patches_dir = Static_patches_dir

    save_dir = os.path.join(base_patches_dir, f"{os.path.splitext(file)[0]}_patches")

    if preprocesspatches:
        patches_of_interest = masked_rail_patches
    else:
        patches_of_interest = original_rail_patches

    if specific_y==None:
        if os.path.exists(save_dir):
            shutil.rmtree(save_dir)  # deletes folder and all contents
        os.makedirs(save_dir)
        

        
        for patch, (x_center, y_center, angle), side, this_track_id in zip(patches_of_interest, patch_positions, patch_sides, patch_track_ids):
            side_str   = (side or "").lower() if isinstance(side, str) else ""
            side_label = "left" if "left" in side_str else ("right" if "right" in side_str else "unknown")
            patch_filename = f"patch_{x_center}_{y_center}_{angle:.3f}_{this_track_id}_{side_label}.png"

            # patch_filename = f"patch_{x_center}_{y_center}_{angle:.3f}_{this_track_id}.png"
            patch_path = os.path.join(save_dir, patch_filename)
            plt.imsave(patch_path, patch)
    
    else:
        left_dir = os.path.join(save_dir, "left")
        right_dir = os.path.join(save_dir, "right")
        unknown_dir = os.path.join(save_dir, "unknown")

        # Clean and recreate the base folder
        if os.path.exists(save_dir):
            shutil.rmtree(save_dir)

        os.makedirs(left_dir)
        os.makedirs(right_dir)
        os.makedirs(unknown_dir)


        for patch, (x_center, y_center, angle), side, this_track_id in zip(patches_of_interest, patch_positions, patch_sides, patch_track_ids):
            side_label = 'left' if 'left' in side.lower() else 'right' if 'right' in side.lower() else 'unknown'

            if side_label == 'left':
                save_path = left_dir
            elif side_label == 'right':
                save_path = right_dir
            else:
                save_path = unknown_dir

            patch_filename = f"patch_{x_center}_{y_center}_{angle:.3f}_{this_track_id}.png"
            patch_path = os.path.join(save_path, patch_filename)

            plt.imsave(patch_path, patch)



    if idx % print_thresh == 0:
        print(f"Saved {len(patches_of_interest)} patches to {save_dir}")

    # === Display some patches side-by-side for comparison ===
    num_patches_to_show = min(5, len(original_rail_patches))  # Show up to 10 patches

    

    if idx % print_thresh == 0:
            
        if preprocesspatches:
            fig, axs = plt.subplots(num_patches_to_show, 2, figsize=(6, 3*num_patches_to_show))
            if num_patches_to_show == 1:
                axs = np.expand_dims(axs, axis=0)

            for i in range(num_patches_to_show):
                axs[i, 0].imshow(original_rail_patches[i])
                axs[i, 0].set_title(f"Original Patch {i}")
                axs[i, 0].axis('off')

                axs[i, 1].imshow(patches_of_interest[i])
                axs[i, 1].set_title(f"Masked Patch {i}")
                axs[i, 1].axis('off')
        else:
            fig, axs = plt.subplots(1, num_patches_to_show, figsize=(3*num_patches_to_show, 3))
            if num_patches_to_show == 1:
                axs = [axs] 

            for i in range(num_patches_to_show):
                axs[i].imshow(original_rail_patches[i])
                axs[i].set_title(f"Original Patch {i}")
                axs[i].axis('off')

        plt.tight_layout()
        plt.show()


    
    # Show patch positions and orientations on the original image
    if idx % print_thresh == 0:
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(original_image)
        ax.set_title("Patch Positions on Original Image",  fontsize=18)
        ax.tick_params(axis='both', labelsize=16)
        for (x, y, angle) in patch_positions:
            # Draw a rotated rectangle to show patch orientation
            rect = plt.Rectangle((x - patch_size//2, y - patch_size//2), 
                                patch_size, patch_size,
                                angle=angle-90,
                                rotation_point='center',
                                edgecolor="red", linewidth=1.5, facecolor="none")
            ax.add_patch(rect)
            
            # Add a line showing the direction
            line_length = patch_size // 2
            dx = line_length * np.cos(np.radians(angle))
            dy = line_length * np.sin(np.radians(angle))
            ax.arrow(x, y, dx, dy, head_width=5, head_length=7, fc='yellow', ec='yellow')
        
        # plt.savefig('patches_1.pdf', format='pdf', bbox_inches='tight')
        plt.show()
        plt.close('all')



# Datasets

We will create two different datasets. One that takes into account that data come from video frames (thus are sequential) and one that does not. For that reason some changes corresponding to each dataset are going to be made

## Sequential Dataset

Check the precentage of not moving photos in that folder to see which folders to exclude from sequential data (print only non zero percentages):

In [ ]:
#define the key for duplicates
subset_cols = [
    'longitude','latitude','tag','type','name',
    'object_uid','sensor','height','width',
    'dataset','label_type','railSide'
]

#mask for 2nd+ occurrences only
dup_after_first = df_track_rgbchigh.duplicated(subset=subset_cols, keep='first')

#count extra-duplicates per tag
dup_counts = df_track_rgbchigh.loc[dup_after_first].groupby('tag').size()

#count all rows per tag
total_counts = df_track_rgbchigh.groupby('tag').size()

#build a DataFrame, fill missing tags with 0, compute %
dup_pct = (
    total_counts
    .to_frame('total')
    .join(dup_counts.to_frame('num_dup'), how='left')
    .fillna(0)
)
dup_pct['dup_pct'] = (dup_pct['num_dup'] / dup_pct['total']) * 100

#round 
dup_pct['dup_pct'] = dup_pct['dup_pct'].round(2)

dup_pct_nonzero = dup_pct[dup_pct['num_dup'] > 0].reset_index()

print(dup_pct_nonzero) #print only tags with duplicates


Keep only stations with moving train (to have the seqiential element):

In [ ]:
threshold = 10                                  
folders_to_keep = dup_pct[dup_pct['dup_pct'] < threshold].index.tolist()
len(folders_to_keep)

For timeseries this dataset because we dont want to remove photos with less occlusion since only one point is needed for time series and no point can be missed. Also we do not use the filtered dataset based on long and lat because 1_calibration_1.2 has duplicates because of moving too slow, but since it is moving we kept it. That is why here df_track_rgbchigh is used and not df_track_rgbchigh_filtered2

In [ ]:
df_track_rgbchigh_filtered_ts = df_track_rgbchigh[df_track_rgbchigh['tag'].isin(folders_to_keep)].copy() 


In [ ]:
df_track_rgbchigh_filtered_ts.shape

#### Now we use functions to crop the patches from original images. First we do not preprocess the backround of the patches at all for comparisson reasons. (sequential dataset)

The horizotal crop limit (750, 3250) is set based on the images printed before and where the tracks we are interested bout are located. However only for one station we observed that the right rail is out of those bounds so we included in the function the limit for that station to be 4000:

In [ ]:
print_image(df_track_rgbchigh_filtered_ts, rail_data_path, '1_calibration_1.2/rgb_highres_center/099_1631441724.000000013.png', with_polyline=True,thresh=1000, target=False)

In [ ]:
Sequential_patches_dir = "./sequential/sequential_noprep/patches_ts_no_prep"

In [ ]:
unique_files = df_track_rgbchigh_filtered_ts.path.unique() #gives unique paths (each representing one photo) 


# for 5 random photos add:[random.sample(range(100), 5)] 

# for doing it only for one station: unique_files = df_track_rgbchigh_filtered_ts[df_track_rgbchigh_filtered_ts['tag']=='1_calibration_1.2'].path.unique()

# for doing it only for specific stations: unique_files = df_track_rgbchigh_filtered_ts[
#     df_track_rgbchigh_filtered_ts['tag'].isin([
#         # '14_signals_station_14.1'#,
#         # '11_main_station_11.1',
#         # '3_fire_site_3.2',
#         '7_approach_underground_station_7.2'
#     ])
# ].path.unique()

idx=0
for file in tqdm(unique_files, desc="Processing images"):
    # print(f"\n=== Processing: {file} ===")
    process_rail_patches(
        df_track_rgbchigh_filtered_ts.sort_values(["path", "trackID", "railSide"]),
        rail_data_path,
        file,
        patch_size=64,
        thresh=300,
        vertical_crop_limits = (1549,2231),  #based on plot of rail distance vs y coordinate 
        idx=idx,
        horizontal_crop_limits=(750, 3250), # based on observation for each station (only for 1_calibration_1.2 the limit to be 4000)
        num_points=21,
        print_thresh=10,
        specific_y=2200,
        preprocesspatches=False
    )

    if idx%10==0:
        print(f"Processed {idx+1} out of {len(unique_files)} images")
    idx=idx+1

##### Move the created patches in another folder and creating their names such that we can connected back to the image from which they were originally cropped

In [ ]:
def unfold_patch_folders(source_dir, target_dir, verbose=True):
    """
    Flatten sequential patch folders into a single target directory.
    Filenames get renamed as: station_patchfolder_basename[_side].png

    Args:
        source_dir (str): Root directory with sequential patch folders.
        target_dir (str): Destination directory to copy flattened patches.
        verbose (bool): Print progress if True.
    """
    os.makedirs(target_dir, exist_ok=True)

    SIDES = {'left', 'right', 'unknown'}
    copied = 0

    for root, _, files in os.walk(source_dir):
        rel = os.path.relpath(root, source_dir)
        parts = rel.split(os.sep)
        if len(parts) < 2:
            continue  # skip top-level

        station = parts[0]

        # decide if current folder is side-specific
        if parts[-1] in SIDES:
            side = parts[-1]
            patch_folder = parts[-2]
        else:
            side = ''
            patch_folder = parts[-1]

        for fn in files:
            if not fn.lower().endswith('.png'):
                continue
            base, ext = os.path.splitext(fn)

            if side == '':
                new_name = f"{station}_{patch_folder}_{base}{ext}"
            else:
                new_name = f"{station}_{patch_folder}_{base}_{side}{ext}"

            src = os.path.join(root, fn)
            dst = os.path.join(target_dir, new_name)
            shutil.copy2(src, dst)
            copied += 1

            if verbose:
                print(f"Copied: {fn} → {new_name}")

    if verbose:
        print(f"\nTotal PNG files copied: {copied}")

    return copied


In [ ]:
source_dir = r"./sequential/sequential_noprep/patches_ts_no_prep"
target_dir = r"./sequential/sequential_noprep/patches_ts_no_prep_unfolder"

n = unfold_patch_folders(source_dir, target_dir)
print(f"Done. {n} PNGs copied.")

##### Seperate the patches in sequences

In [ ]:

def _win_long(p: str) -> str:
    if os.name == "nt":
        ap = os.path.abspath(p)
        return ap if ap.startswith("\\\\?\\") else "\\\\?\\" + ap
    return p

def create_track_sequences(input_dir, output_dir):
    """
    Organize rail patches into sequences based on station, track ID, and side.
    Patches that cannot be assigned are moved to an "unassigned" folder.

    Args:
        input_dir (str): Directory containing the input patch images.
        output_dir (str): Directory where the organized sequences will be saved.
    """
    # Handle long paths on Windows
    input_dir  = _win_long(input_dir)    
    output_dir = _win_long(output_dir)   

    unassigned_dir = os.path.join(output_dir, "unassigned")

    pattern = re.compile(
        r'^(?P<station>.+?)_'                       # Station 
        r'(?P<photo>\d+_\d+\.\d+)_'                 # Photo ID (e.g. 000_1631441714.100000026)
        r'patches_patch_'                           # "patches_patch_"
        r'(?P<coords>\d+_\d+_[\d\.]+)_'             # Coordinates & angle (e.g. 2329_2199_83.570)
        r'(?P<track>-?\d+)_'                        # Track ID (e.g. -2 or 3)
        r'(?P<side>(?:left|right))'                 # Side (left or right)
        r'(?:\..+)?$'                               
    )

    # Create the "unassigned" folder (inside output_dir) if it doesn’t exist
    os.makedirs(unassigned_dir, exist_ok=True)

    # Gather all files in the input directory
    try:
        all_files = [
            f for f in os.listdir(input_dir)
            if os.path.isfile(os.path.join(input_dir, f))
        ]
    except FileNotFoundError:
        raise SystemExit(f"ERROR: Input folder '{input_dir}' does not exist. Please check path.")

    # group patches by station, then later by (track, side)
    patches_by_station = defaultdict(list)
    unassigned = []

    # Parse every filename 
    for filename in all_files:
        match = pattern.match(filename)
        if not match:
            # Filename doesn’t match the expected pattern → move to unassigned
            unassigned.append(filename)
            continue

        station = match.group("station")
        photo = match.group("photo")
        track = match.group("track")
        side = match.group("side")
        # Extract the “image number” (the integer prefix before the underscore in photo)
        image_number = int(photo.split("_")[0])

        patches_by_station[station].append({
            "file": filename,
            "photo": photo,
            "track": track,
            "side": side,
            "image_number": image_number
        })

    # For each station, build sequences per (track, side) 
    for station, patches in patches_by_station.items():
        # Find every distinct (track, side) pair for this station
        track_side_pairs = set((p["track"], p["side"]) for p in patches)

        # Make a subfolder for this station
        station_dir = os.path.join(output_dir, station)
        os.makedirs(station_dir, exist_ok=True)

        # For each distinct (track, side), create its own sequence folder
        for (track, side) in track_side_pairs:
            seq_dir = os.path.join(station_dir, f"{track}_{side}")
            os.makedirs(seq_dir, exist_ok=True)

            # Keep track of which image_numbers we’ve already used in this sequence
            used_numbers = set()

            # Filter patches that belong to (track, side)
            relevant_patches = [
                p for p in patches
                if (p["track"], p["side"]) == (track, side)
            ]
            # Sort them by image_number ascending
            relevant_patches.sort(key=lambda x: x["image_number"])

            # Copy only one patch per image_number into seq_dir
            for p in relevant_patches:

                src = os.path.join(input_dir, p["file"])
                dst = os.path.join(seq_dir, p["file"])

                # ensure parent exists & wrap long paths  
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                src = _win_long(src)
                dst = _win_long(dst)

                if p["image_number"] not in used_numbers:
                    src = os.path.join(input_dir, p["file"])
                    dst = os.path.join(seq_dir, p["file"])
                    shutil.copy2(src, dst)
                    used_numbers.add(p["image_number"])
                else:
                    # If this image_number is already used, divert to unassigned
                    unassigned.append(p["file"])

    # Move all "unassigned" files into the single unassigned folder
    for filename in unassigned:
        src = os.path.join(input_dir, filename)
        dst = os.path.join(unassigned_dir, filename)

        os.makedirs(os.path.dirname(dst), exist_ok=True)  
        src = _win_long(src)                               
        dst = _win_long(dst)   

        shutil.copy2(src, dst)

    # Print a summary of unassigned files 
    if unassigned:
        print(f"The following files were unassigned (moved to '{unassigned_dir}'):")
        for f in unassigned:
            print(f"  - {f}")
    else:
        print("All files were assigned to sequences successfully!")


In [ ]:
input_dir = r"./sequential/sequential_noprep/patches_ts_no_prep_unfolder"
output_dir = r"./sequential/sequential_noprep/patches_ts_no_prep_sequencesFolders"

create_track_sequences(input_dir, output_dir)

##### Check if all assigned correctly

In [ ]:
IMAGE_EXTENSIONS = {'.png'} 

def _win_long(p: str) -> str:
    if os.name == "nt":
        ap = os.path.abspath(p)
        return ap if ap.startswith("\\\\?\\") else "\\\\?\\" + ap
    return p

def count_images_in_subfolders(base_dir: str, use_long_paths: bool = True):
    """
    For each immediate subfolder in base_dir, iterate its immediate subfolders
    and print how many image files each contains.
    """
    if use_long_paths:
        base_dir = _win_long(base_dir)

    if not os.path.isdir(base_dir):
        print(f"Error: directory not found:\n  {base_dir}")
        return

    # sort by folder name (case-insensitive)
    parents = sorted((e for e in os.scandir(base_dir) if e.is_dir()),
                     key=lambda e: e.name.lower())

    for parent in parents:
        print(f"Folder: {parent.name}")
        any_sub = False

        subs = sorted((e for e in os.scandir(parent.path) if e.is_dir()),
                      key=lambda e: e.name.lower())

        for sub in subs:
            any_sub = True
            count = 0
            for f in os.scandir(sub.path):
                if f.is_file():
                    _, ext = os.path.splitext(f.name)
                    if ext.lower() in IMAGE_EXTENSIONS:
                        count += 1
            print(f"    {sub.name}: {count} images")

        if not any_sub:
            print("    (no subfolders)")
        print()


In [ ]:
base_dir =r"./sequential/sequential_noprep/patches_ts_no_prep_sequencesFolders"  
count_images_in_subfolders(base_dir)

### Next, we remove the backround from the same patches we cropped before and save them 

In [ ]:
Sequential_patches_dir = "./sequential/patches_ts_black"

In [ ]:
unique_files = df_track_rgbchigh_filtered_ts.path.unique() #gives unique paths (each representing one photo) 

idx=0
for file in tqdm(unique_files, desc="Processing images"):
    # print(f"\n=== Processing: {file} ===")
    process_rail_patches(
        df_track_rgbchigh_filtered_ts.sort_values(["path", "trackID", "railSide"]),
        rail_data_path,
        file,
        patch_size=64,
        thresh=300,
        vertical_crop_limits = (1549,2231),  #based on plot of rail distance vs y coordinate 
        idx=idx,
        horizontal_crop_limits=(750, 3250), # based on observation for each station (only for 1_calibration_1.2 the limit to be 4000)
        num_points=21,
        print_thresh=10,
        specific_y=2200
    )

    if idx%10==0:
        print(f"Processed {idx+1} out of {len(unique_files)} images")
    idx=idx+1

#### Move the created patches in another folder (creating their names such that we can connected back to the image from which they were originally cropped)

In [ ]:
source_dir = r"./sequential/patches_ts_black"
target_dir = r"./sequential/patches_ts_black_unfolder"

n = unfold_patch_folders(source_dir, target_dir)
print(f"Done. {n} PNGs copied.")

#### Second step of prprocessing (remove more backround)

Since the points based on which we cropped the patches and creted the masks were not located in the two edges we still have some minor backround left, which we wanted to eliminate. For this we used K-means clustering with the darker, mid and brighter colors in each patch since the rail track part is mostly brighter than the backround. 

First we found the density of the three clusters across all images. 

In [ ]:
def abs_mid_bright_finder(input_dir):
    input_dir = input_dir #"./train_rail__before_final_preprocessing"   
    n_clusters = 3 #bright,mid,dark                            

    #Collect all cluster‐centers
    centers_list = []
    for path in glob(os.path.join(input_dir, "*.png")):
        gray = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2GRAY)
        pixels = gray.reshape(-1,1)
        pixels = pixels[pixels[:,0] != 0]   # drop pure‐black
        
        km = KMeans(n_clusters=n_clusters, random_state=0).fit(pixels)
        # sort so each row = [dark, mid, bright]
        centers_sorted = np.sort(km.cluster_centers_.flatten())
        centers_list.append(centers_sorted)

    centers_arr = np.vstack(centers_list)
    centers_dark   = centers_arr[:,0]
    centers_mid    = centers_arr[:,1]
    centers_bright = centers_arr[:,2]

    # # Plot histograms
    # plt.figure(figsize=(8,4))
    # plt.hist(centers_dark,   bins=50, alpha=0.6, label="dark")
    # plt.hist(centers_mid,    bins=50, alpha=0.6, label="mid")
    # plt.hist(centers_bright, bins=50, alpha=0.6, label="bright")
    # plt.title("Distribution of per‐patch k-means cluster centers")
    # plt.xlabel("Grey‐value")
    # plt.legend()
    # plt.tight_layout()
    # plt.show()

    # Fit a 3-component GMM to *all* centers
    all_centers = centers_arr.flatten().reshape(-1,1)
    gmm = GaussianMixture(n_components=3, random_state=0).fit(all_centers) #3: bright,mid,dark     

    # sort the components by their mean μ
    order = np.argsort(gmm.means_.flatten())
    mus    = gmm.means_.flatten()[order]
    vars_  = gmm.covariances_.flatten()[order]
    weights= gmm.weights_.flatten()[order]

    def comp_pdf(x, mu, var, w): #weighted Gaussian (normal) probability density,
        return w * (1.0/np.sqrt(2*np.pi*var)) * np.exp(-0.5*((x-mu)**2)/var)

    #root finder method: brentq
    #find a solution lambda x: comp_pdf(x, mus[0], vars_[0], weights[0])- comp_pdf(x, mus[1], vars_[1], weights[1])=0, somewhere between mu_0 and mu_1:
    x12 = brentq(
        lambda x: comp_pdf(x, mus[0], vars_[0], weights[0])
                - comp_pdf(x, mus[1], vars_[1], weights[1]),
        mus[0], mus[1]
    )
    #find a solution lambda x: comp_pdf(x, mus[1], vars_[1], weights[1]) - comp_pdf(x, mus[2], vars_[2], weights[2]), somewhere between mu_1 and mu_2:
    x23 = brentq(
        lambda x: comp_pdf(x, mus[1], vars_[1], weights[1])
                - comp_pdf(x, mus[2], vars_[2], weights[2]),
        mus[1], mus[2]
    )

    print(f"Cluster‐center modes (μ):    {mus}")
    print(f"Valley between mode1/2:     {x12:.1f}")
    print(f"Valley between mode2/3:     {x23:.1f}")

    #thresholds
    ABS_MID    = x12   
    ABS_BRIGHT = x23  

    print(f"\n→ Use ABS_MID = {ABS_MID:.0f}, ABS_BRIGHT = {ABS_BRIGHT:.0f}")

    plt.figure(figsize=(10,4))

    #density histograms
    plt.hist(centers_dark,   bins=50, alpha=0.4, label="dark",   density=True)
    plt.hist(centers_mid,    bins=50, alpha=0.4, label="mid",    density=True)
    plt.hist(centers_bright, bins=50, alpha=0.4, label="bright", density=True)

    #x-axis over the grey‐value range
    x = np.linspace(centers_arr.min(), centers_arr.max(), 1000)

    #each GMM component curve
    for mu, var, w, col, lab in zip(
            mus, vars_, weights,
            ["C0","C1","C2"],
            ["GMM dark", "GMM mid", "GMM bright"]):
        y = comp_pdf(x, mu, var, w)
        plt.plot(x, y, color=col, linewidth=2, label=lab)

    #valley lines too
    plt.axvline(x12, color="k", linestyle="--", label="valley dark↔mid")
    plt.axvline(x23, color="k", linestyle="-.", label="valley mid↔bright")

    plt.title("Brightness-Cluster Thresholds for Rail Patch Background Removal",fontsize=18)
    plt.xlabel("Grey‐value",fontsize=17)
    plt.ylabel("Density",fontsize=17)
    plt.legend(fontsize=15)
    plt.tight_layout()
    plt.tick_params(axis='both', labelsize=16)
    plt.savefig('valleys_sequentail.png', format='png', bbox_inches='tight')
    plt.show()

    return ABS_MID,ABS_BRIGHT



In [ ]:
ABS_MID1,ABS_BRIGHT1 = abs_mid_bright_finder(input_dir="./sequential/patches_ts_black_unfolder")

Function for removind the backround and save the new created patches:

In [ ]:
def additional_preprocessing(input_dir, save_dir, thresh=100, ABS_BRIGHT=239.1,ABS_MID = 60.1):
    os.makedirs(save_dir, exist_ok=True)
    image_paths = glob(os.path.join(input_dir, "*.png"))
    for idx, full_path in enumerate(tqdm(image_paths, desc="Processing images")):
        img_bgr = cv2.imread(full_path)
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        gray    = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

        # K‐means clustering on gray image
        pixels         = gray.reshape(-1, 1)
        non_black_mask = (pixels[:, 0] != 0) # drop pure-black background
        pixels_nb      = pixels[non_black_mask]
        km = KMeans(n_clusters=3, random_state=0).fit(pixels_nb)
        centers = km.cluster_centers_.flatten() # the three centroids
        counts  = np.bincount(km.labels_, minlength=3)

        order             = np.argsort(centers)
        centers_sorted    = centers[order]
        percentages_sorted = (counts[order] / float(counts.sum())) * 100.0

        c_dark, c_mid, c_bright     = centers_sorted
        p_dark, p_mid, p_bright     = percentages_sorted

        # pick whether a cluster is ll the rail track by absolute center, then by percentage 
        ABS_BRIGHT = ABS_BRIGHT 
        ABS_MID    = ABS_MID 

        if c_bright >= ABS_BRIGHT:
            chosen_sorted_idx = 2 #bright
        elif c_mid >= ABS_MID:
            chosen_sorted_idx = 1 #mid
        else:
            # fallback to bigger of the two “potential rail” clusters
            chosen_sorted_idx = 2 if (p_bright > p_mid) else 1

        #build a threshold halfway between that cluster’s center and the next‐lower center:
        c_low   = centers_sorted[chosen_sorted_idx - 1]
        c_high  = centers_sorted[chosen_sorted_idx]
        n_low   = int(counts[order[chosen_sorted_idx - 1]]) #how many pixels fell into this cluster
        n_high  = int(counts[order[chosen_sorted_idx]])
        threshold = float((c_low * n_low + c_high * n_high) / float(n_low + n_high)) #weighted average of the two centroids

        # Threshold the gray image to get only “rail” pixels.
        _, mask_track = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)

        col_sum = np.sum(mask_track > 0, axis=0)  #count of white (rail) pixels in each column
        col_height_thresh = gray.shape[0] * 0.10  #image height* 0.10  
        good_cols = np.where(col_sum >= col_height_thresh)[0] #indices of columns whose white‐pixel count meets or exceeds that 10% threshold.


        final = np.zeros_like(img_rgb)  # black image 
        if good_cols.size > 0:

            #crop vertical band
            x_min, x_max = int(good_cols.min()), int(good_cols.max())

            # (pad by a couple of pixels on each side) :
            pad = 2
            x_min = max(0,   x_min - pad)
            x_max = min(gray.shape[1] - 1, x_max + pad)

            # copy the full‐height vertical band:
            final[:, x_min:(x_max + 1)] = img_rgb[:, x_min:(x_max + 1)]

            # draw a rectangle for visual check: ———
            track_rect = img_rgb.copy()
            cv2.rectangle(
                track_rect,
                (x_min, 0),
                (x_max, gray.shape[0] - 1),
                (255, 165, 0), 3
            )

        else: #If no “good” columns are found, it instead finds contours in the binary mask and crops the bounding box of the largest one
            print('no good columns found, check contours')
            contours, _ = cv2.findContours(mask_track, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if contours:
                print('contours found')
                largest = max(contours, key=cv2.contourArea)
                x2, y2, w2, h2 = cv2.boundingRect(largest)
                final[y2:y2+h2, x2:x2+w2] = img_rgb[y2:y2+h2, x2:x2+w2]
                track_rect = img_rgb.copy()
                cv2.rectangle(track_rect, (x2, y2), (x2 + w2 - 1, y2 + h2 - 1), (255, 165, 0), 3)
            else:
                print('no contours found')
                track_rect = img_rgb.copy()
                # no contour → no rectangle

        # Save ‘final’ 
        base_name = os.path.splitext(os.path.basename(full_path))[0]
        out_path  = os.path.join(save_dir, f"{base_name}.png")
        cv2.imwrite(out_path, cv2.cvtColor(final, cv2.COLOR_RGB2BGR))

        # show a few plots 
        if idx % thresh == 0:
            print(full_path)
            fig, ax = plt.subplots(1, 3, figsize=(12, 4))
            ax[0].imshow(img_rgb);        ax[0].set_title("Original",fontsize=18);      ax[0].axis("off")
            ax[1].imshow(track_rect);     ax[1].set_title("Track rect only",fontsize=18); ax[1].axis("off")
            ax[2].imshow(final);          ax[2].set_title("Final output",fontsize=18);   ax[2].axis("off")
            plt.tight_layout()
            # if idx==1400:
            #     fig.savefig(f"output_{idx}.pdf" , format="pdf", bbox_inches="tight", pad_inches=0.05)
            plt.show()

In [ ]:
additional_preprocessing(input_dir = "./sequential/patches_ts_black_unfolder" , save_dir = "./sequential/patches_ts_black_prep2", ABS_MID = ABS_MID1,ABS_BRIGHT = ABS_BRIGHT1) 

#### Seperate data to sequences

In [ ]:
input_dir = r"./sequential/patches_ts_black_prep2"
output_dir = r"./sequential/patches_ts_black_prep2_sequencesFolders"

create_track_sequences(input_dir, output_dir)

#### Check if all assigned correcty

In [ ]:
base_dir =r"./sequential/patches_ts_black_prep2_sequencesFolders"  
count_images_in_subfolders(base_dir)

## Static dataset (more data but not cropping them sequentially)-->only the black backround version 

### Mask images and crop patches

In [ ]:
Static_patches_dir = "./static/patches_static"

In [ ]:
unique_files = df_track_rgbchigh_filtered2.path.unique()#[random.sample(range(100), 5)] #gives unique paths (each representing one photo)


idx=0
for file in tqdm(unique_files, desc="Processing images"):
    # print(f"\n=== Processing: {file} ===")
    process_rail_patches(
        df_track_rgbchigh_filtered2.sort_values(["path", "trackID", "railSide"]),
        rail_data_path,
        file,
        patch_size=64,
        thresh=300,
        vertical_crop_limits = (1549,2231),  #based on plot of rail distance vs y coordinate
        idx=idx,
        horizontal_crop_limits=(750, 3250),
        num_points=21,
        print_thresh=100
    )

    if idx%100==0:
        print(f"Processed {idx+1} out of {len(unique_files)} images")
    idx=idx+1

### Move the created patches in another folder and creating their names such that we can connected back to the image from which they were originally cropped


In [ ]:
source_dir = "./static/patches_static"
target_dir = "./static/patches_static_unfolder"


os.makedirs(target_dir, exist_ok=True)

SIDES = {'left', 'right', 'unknown'}
copied = 0

for root, _, files in os.walk(source_dir):
    # relative path under 'patches_timeseries'
    rel = os.path.relpath(root, source_dir)
    parts = rel.split(os.sep)
    if len(parts) < 2:
        # nothing to do at the very top level
        continue

    station = parts[0]

    # if the folder we're in _is_ a side‐folder, grab it; otherwise default
    if parts[-1] in SIDES:
        side = parts[-1]
        patch_folder = parts[-2]
    else:
        side = ''
        patch_folder = parts[-1]

    for fn in files:
        if not fn.lower().endswith('.png'):
            continue
        base, ext = os.path.splitext(fn)
        if side == '':
            new_name = f"{station}_{patch_folder}_{base}{ext}"
        else:
            new_name = f"{station}_{patch_folder}_{base}_{side}{ext}"
        src = os.path.join(root, fn)
        dst = os.path.join(target_dir, new_name)
        shutil.copy2(src, dst)
        copied += 1
        print(f"Copied: {fn} → {new_name}")

print(f"\nTotal PNG files copied: {copied}")


### Second step of prprocessing (remove more backround)-->same procedure as before

In [ ]:
ABS_MID2,ABS_BRIGHT2 = abs_mid_bright_finder(input_dir="./static/patches_static_unfolder")

### Remove Backround further (as before)

In [ ]:
additional_preprocessing(input_dir = "./static/patches_static_unfolder" , save_dir = "./static/patches_static_prep2",ABS_BRIGHT=ABS_BRIGHT2,ABS_MID =ABS_MID2)

# Instead of making backround black, we also make it blur using the black backround mask

In [ ]:
def list_images(folder, exts):
    files = []
    for ext in exts:
        files.extend(folder.rglob(f"*.{ext}"))
        files.extend(folder.rglob(f"*.{ext.upper()}"))
    return sorted(files)

def build_mask_from_black(img, threshold=0, use_gray=False):
    """
    Return uint8 mask (0/255) where pixels are considered black.
    """
    if use_gray:
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        mask = (gray <= threshold)
    else:
        b,g,r = cv2.split(img)
        mask = (b<=threshold) & (g<=threshold) & (r<=threshold)
    return (mask.astype(np.uint8))*255

def blur_where_mask(img, mask, ksize=15, sigma=0, dilate_iters=1, erode_iters=0):
    """
    Gaussian-blur only where mask==255, blend over original.
    """
    if ksize%2==0 or ksize<3:
        raise ValueError("kernel size must be odd and >=3")
    work_mask = mask.copy()
    if dilate_iters>0:
        work_mask = cv2.dilate(work_mask, np.ones((3,3), np.uint8), iterations=dilate_iters)
    if erode_iters>0:
        work_mask = cv2.erode(work_mask, np.ones((3,3), np.uint8), iterations=erode_iters)
    blurred = cv2.GaussianBlur(img, (ksize, ksize), sigma)
    inv = cv2.bitwise_not(work_mask)
    bg = cv2.bitwise_and(blurred, blurred, mask=work_mask)
    fg = cv2.bitwise_and(img, img, mask=inv)
    return cv2.add(bg, fg), work_mask


In [ ]:
folderA = Path("./sequential/patches_ts_black_prep2")                                  # images where background is black (mask source)
folderB = Path("./sequential/sequential_noprep/patches_ts_no_prep_unfolder")        # original patches that we want to blur their background
out_dir = Path("./sequential/blur_backround_images")                                # where to save blurred results

# Masking/blur parameters
threshold = 0       # only fully black pixels
use_gray = False    # True -> grayscale threshold, False -> all RGB channels 
kernel = 15         # odd integer >= 3 (e.g., 11, 15, 21)
sigma = 0.0       
dilate_iters = 1    # expand mask to soften edge (0 to disable)
erode_iters = 0     # shrink mask after dilate
strict_size = True  # if True, skip size-mismatched pairs; if False, resize B to A
exts = ["png","jpg","jpeg","bmp","tiff"]  # extensions to match


A_list = list_images(folderA, exts)
B_list = list_images(folderB, exts)

A_map = {p.name: p for p in A_list}
B_map = {p.name: p for p in B_list}

common = sorted(set(A_map.keys()) & set(B_map.keys()))
print(f"Found {len(common)} matching filenames.")

out_dir.mkdir(parents=True, exist_ok=True)

processed = 0
skipped_size = 0

for name in common:
    pa, pb = A_map[name], B_map[name]
    imgA = cv2.imread(str(pa), cv2.IMREAD_COLOR)
    imgB = cv2.imread(str(pb), cv2.IMREAD_COLOR)
    if imgA is None or imgB is None:
        print(f"[WARN] Could not read: {name}. Skipping.")
        continue

    hA,wA = imgA.shape[:2]
    hB,wB = imgB.shape[:2]
    if (hA,wA)!=(hB,wB):
        if strict_size:
            print(f"[WARN] Size mismatch for {name}: A={wA}x{hA}, B={wB}x{hB}. Skipping.")
            skipped_size += 1
            continue
        else:
            imgB = cv2.resize(imgB, (wA,hA), interpolation=cv2.INTER_AREA)

    mask = build_mask_from_black(imgA, threshold=threshold, use_gray=use_gray)
    out, used_mask = blur_where_mask(
        imgB, mask, ksize=kernel, sigma=sigma,
        dilate_iters=dilate_iters, erode_iters=erode_iters
    )

    ok = cv2.imwrite(str(out_dir / name), out)
    if not ok:
        print(f"[WARN] Failed to save: {name}")
        continue

    processed += 1

print(f"Done. Processed {processed} images. Skipped size mismatches: {skipped_size}. Output -> {out_dir}")




## Show an example

In [ ]:
if common:
    name = random.choice(common)
    pa, pb = A_map[name], B_map[name]
    imgA = cv2.imread(str(pa), cv2.IMREAD_COLOR)
    imgB = cv2.imread(str(pb), cv2.IMREAD_COLOR)
    if imgA is not None and imgB is not None:
        if imgA.shape[:2] != imgB.shape[:2]:
            imgB = cv2.resize(imgB, (imgA.shape[1], imgA.shape[0]), interpolation=cv2.INTER_AREA)
        mask = build_mask_from_black(imgA, threshold=threshold, use_gray=use_gray)
        out, used_mask = blur_where_mask(imgB, mask, ksize=kernel, sigma=sigma,
                                         dilate_iters=dilate_iters, erode_iters=erode_iters)

        # Convert BGR->RGB for display
        imgB_rgb = cv2.cvtColor(imgB, cv2.COLOR_BGR2RGB)
        out_rgb = cv2.cvtColor(out, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(14,4))
        plt.subplot(1,3,1); plt.title("Original B"); plt.imshow(imgB_rgb); plt.axis('off')
        plt.subplot(1,3,2); plt.title("Mask from A"); plt.imshow(used_mask, cmap='gray'); plt.axis('off')
        plt.subplot(1,3,3); plt.title("Blurred (masked)"); plt.imshow(out_rgb); plt.axis('off')
        plt.suptitle(name)
        plt.show()
else:
    print("No common files to preview.")

# Applies CLAHE only on the rail region (based on where the black version is non-black) in the luminance (L) channel.

In [ ]:
def apply_clahe_to_rail(rail_bgr, black_bgr, clipLimit=2.0, tileGridSize=(8,8)):
    """
    Given:
      - rail_bgr: BGR patch with blur BG (or original) [H,W,3]
      - black_bgr: BGR patch with black BG (so background is black, rail region nonzero)
    Returns:
      - out_bgr: BGR image where CLAHE has been applied on the rail (luminance) region (not on balckround)
    """
    # Create mask: where black_bgr is non-black across channels  
    # We assume background is (0,0,0), so any pixel > 0 in any channel is part of rail
    mask = np.any(black_bgr > 0, axis=2).astype(np.uint8)  # 1 = rail, 0 = background

    # Convert the rail_bgr to LAB so we can apply CLAHE on L channel
    lab = cv2.cvtColor(rail_bgr, cv2.COLOR_BGR2LAB)
    L, A, B = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=clipLimit, tileGridSize=tileGridSize)

    # prepare an output L channel
    L_new = L.copy()

    # Apply CLAHE only where mask == 1
    # create an index or mask
    coords = np.where(mask == 1)
    if coords[0].size > 0:
        L_eq = clahe.apply(L)
        L_new[coords] = L_eq[coords]

    # Merge back
    lab_new = cv2.merge((L_new, A, B))
    out_bgr = cv2.cvtColor(lab_new, cv2.COLOR_LAB2BGR)
    return out_bgr

def process_dirs(dir1, black_dir, output_dir, clipLimit=2.0, tileGridSize=(8,8)):
    """
    For each file in dir1 (and matching in black_dir),
    open both, apply CLAHE on the rail region, save to output_dir.
    """
    os.makedirs(output_dir, exist_ok=True)
    fnames = sorted(os.listdir(dir1))
    for fname in fnames:
        path_blur = os.path.join(dir1, fname)
        path_black = os.path.join(black_dir, fname)
        if not os.path.isfile(path_blur) or not os.path.isfile(path_black):
            print(f"Skipping {fname}, missing in one of the dirs.")
            continue

        blur_img = cv2.imread(path_blur)
        black_img = cv2.imread(path_black)
        if blur_img is None or black_img is None:
            print(f"Could not read {fname} in one of dirs.")
            continue

        # Ensure they have the same shape
        if blur_img.shape != black_img.shape:
            # resize black to match blur (nearest) or skip
            black_img = cv2.resize(black_img, (blur_img.shape[1], blur_img.shape[0]),
                                   interpolation=cv2.INTER_NEAREST)
            # or continue

        out_img = apply_clahe_to_rail(blur_img, black_img, clipLimit=clipLimit, tileGridSize=tileGridSize)

        out_path = os.path.join(output_dir, fname)
        cv2.imwrite(out_path, out_img)

    print("Done. Saved CLAHE outputs to", output_dir)




## Apply to blur images

In [ ]:
process_dirs(dir1 = "./sequential/blur_backround_images" , #folder with all patches with blured backround (unlabeled)
             black_dir = "./sequential/patches_ts_black_prep2",  #folder with all patches with black backround (unlabeled)
             output_dir = "./sequential/blurclahe_images") #folder with all patches with black backround and clahe applied to track region

## Apply to black images

In [ ]:
process_dirs(dir1 = "./sequential/patches_ts_black_prep2" , black_dir = "./sequential/patches_ts_black_prep2", output_dir = "./sequential/blackclahe_images")

# Apply black white to best pipeline from experiment 1(blurclahe)

In [ ]:
def to_monochrome_L(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    for fname in os.listdir(input_dir):
        in_path = os.path.join(input_dir, fname)
        if not os.path.isfile(in_path): 
            continue
        img = cv2.imread(in_path)  # BGR
        if img is None:
            continue
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        L, A, B = cv2.split(lab)
        L3 = cv2.merge([L, L, L])  # 3 channels for model compatibility
        cv2.imwrite(os.path.join(output_dir, fname), L3)




In [ ]:
to_monochrome_L(Path("./sequential/blurclahe_images"), Path("./sequential/blurclahebw_images"))

# Figure with all pipelines

In [ ]:
pipeline_images = [
    ("ORIGINAL",      "original.png"),
    ("BLACK_BG",      "black.png"),
    ("BLACK_BG + CLAHE", "blackclahe.png"),
    ("BLUR_BG",       "blur.png"),
    ("BLUR_BG + CLAHE", "blurclahe.png"),
    ("BLUR_BG + CLAHE + GREY", "grey.png"),
]

fig, axes = plt.subplots(1, 6, figsize=(12, 8))
axes = axes.flatten()

for ax, (title, filename) in zip(axes, pipeline_images):
    if not os.path.exists(filename):
        print(f"File not found: {filename}")
        continue

    img = Image.open(filename)
    ax.imshow(img)
    ax.set_title(title, fontsize=11)
    ax.axis("off")

plt.tight_layout()
plt.savefig("preprocessing_grid.png", dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved as preprocessing_grid.png")
